# Milestone 3 : moteur de consolidation

Ce notebook construit le moteur de consolidation des livraisons : à partir du planning des besoins des usines, il regroupe les lignes dans des camions, choisit pour chaque couple fournisseur × usine ses jours fixes de livraison, lisse la charge des usines sur la semaine et chiffre les économies réalisées par rapport à la situation actuelle (baseline du milestone 2).

### Définition des variables et des fonctions

In [ ]:
import itertools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------- Fichiers ----------
SILVER_CLEAN_PATH = "../data/silver/released_schedule_cleaned.csv"
QUARANTAINE_PATH = "../data/silver/released_schedule_quarantine.csv"
BASELINE_PATH = "../data/baseline/baseline_camion.csv"   # baseline du milestone 2 : une ligne par camion
SORTIE = Path("../data/gold")

# ---------- Paramètres du moteur ----------
POIDS_MAX = 24500                 # kg par camion
VOLUME_MAX = 110.4                # m³ par camion
FENETRE = pd.Timedelta(days=7)    # avance maximale d'une livraison sur le besoin
JOURS_FIXES_MAX = 3               # un couple livre 1 à 3 jours fixes par semaine
SEUIL_LISSAGE = 0.35              # écart max toléré entre un jour et la moyenne de la semaine
MIN_CAMIONS_SEMAINE = 5           # semaines plus petites (moins d'un camion par jour) : pas contrôlées
JOURS = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi"]

# ---------- Coût de transport ----------
FREIGHT_VALUE_MARGIN = 2.3923     # +239,23 %
DIESEL_SURCHARGE = 1.1943
couts = pd.read_csv("../data/bronze/rate_sheet.csv", index_col="km_max")
TRANCHES_KM = [int(km) for km in couts.index if not km.startswith(">")]    # 30, 50, ..., 650
TRANCHES_KG = [int(kg) for kg in couts.columns if not kg.startswith(">")]  # 50, 100, ..., 4000
distances = pd.read_csv("../data/bronze/distances.csv", encoding="utf-8-sig").set_index("supplier_id")["distance"]

# Données manquantes : estimations hautes (comme dans la baseline du milestone 2)
DISTANCE_PAR_DEFAUT = TRANCHES_KM[-1] + 1  # 651 km -> tranche "> 650 km"
POIDS_PAR_DEFAUT = TRANCHES_KG[-1] + 1     # 4 001 kg -> tranche "> 4 000 kg"
SANS_DISTANCE = sorted(distances.index[distances.isna()])
distances = distances.fillna(DISTANCE_PAR_DEFAUT)


def tranche(valeur, tranches):
    # Première tranche >= valeur (on majore), sinon la tranche '>max'
    for borne in tranches:
        if valeur <= borne:
            return str(borne)
    return f">{tranches[-1]}"


def cout_transport(poids, distance):
    # Prix d'un camion en € = coût de la grille × (1 + marge) × diesel
    cout_base = couts.loc[tranche(distance, TRANCHES_KM), tranche(poids, TRANCHES_KG)]
    return float(round(cout_base * (1 + FREIGHT_VALUE_MARGIN) * DIESEL_SURCHARGE, 2))


# ---------- Mise en forme ----------
def nombre(x):
    return f"{x:,.1f}".replace(",", " ").replace(".", ",").removesuffix(",0")


def pourcent(x):
    return f"{x:.1%}".replace(".", ",").replace("%", " %")


def eur(x):
    return f"{x:,.0f} €".replace(",", " ")

## 1. Contraintes et objectif

### Contraintes

Les contraintes dures ne sont jamais violées. La contrainte de lissage est souple : le moteur la poursuit, mais une semaine peut rester hors tolérance quand rien ne permet de l'équilibrer (petites semaines, voir partie 5).

| # | Contrainte | Type | Fonction |
|---|---|---|---|
| C1 | Un camion ne dépasse pas 24,5 t | dure | `c1_poids` |
| C2 | Un camion ne dépasse pas 110,4 m³ | dure | `c2_volume` |
| C3 | Chaque ligne est livrée, une seule fois, entière (lots indivisibles) | dure | `c3_exhaustivite` |
| C4 | Un camion ne transporte qu'un seul couple | dure | `c4_un_couple` |
| C5 | Jamais de retard | dure | `c5_retard` |
| C6 | Au plus 7 jours d'avance | dure | `c6_avance` |
| C7 | Livraison un jour ouvré, dans l'horizon du planning (semaine ajoutée avant le planning comprise) | dure | `c7_jours_ouvres` |
| C8 | Jours fixes : 1 à 3 jours par couple, et chaque camion part un de ces jours, sans exception | dure | `c8_jours_fixes` |
| C9 | Au plus 1 camion par couple et par jour, camions réservés compris ; les cas impossibles (besoin d'un jour > 24,5 t, ligne sans poids) sont notifiés | dure | `c9_un_camion_par_jour` |
| C10 | Lissage : dans chaque usine, aucun jour ne s'écarte de plus de 35 % (et d'au moins 1 camion) de la moyenne de la semaine ; semaines d'au moins 5 camions | souple | `c10_lissage` |
| C11 | Profil utilisé : un couple à 2 ou 3 jours fixes livre au moins une semaine sur tous ses jours fixes | dure | `c11_profil_utilise` |

Deux cas particuliers, fixés par les hypothèses des milestones 1 et 2 :
- début du planning : le planning des besoins commence le lundi 09/11, mais on peut livrer dans la semaine qui le précède (S45, du 02/11 au 06/11), ajoutée à l'horizon. Un besoin du début du planning qui tombe avant le premier jour fixe du couple part donc, comme les autres, le dernier jour fixe qui le précède, au plus 7 jours avant : tous les camions partent un jour fixe (C8).

- lignes de la quarantaine sans poids : elles ne peuvent pas être consolidées. Chacune a son camion réservé, le jour du besoin, chiffré en tranche > 4 000 kg. Cette règle est absolue : un camion réservé n'est jamais regroupé ni déplacé, même s'il tombe hors des jours fixes du couple (C8) ou le même jour qu'un autre camion du couple (C9). Ces cas impossibles ne sont pas cachés : chacun est notifié dans les alertes (fonction `alertes`, fichier `data/gold/alertes.csv`), avec sa cause et l'action à mener. Ces camions sont exemptés de C1 (poids inconnu) et comptent dans la charge des usines (C10).

### Objectif

Minimiser le coût total de transport :

$$\min \sum_{c} \text{coût}(c), \qquad \text{coût}(c) = \text{grille}\big[\text{tranche}(\text{distance}_{k(c)}),\ \text{tranche}(\textstyle\sum_{l \in c} w_l)\big] \times (1 + 2{,}3923) \times 1{,}1943$$

La grille donne un prix par tranche de distance et de poids : un camion plus plein coûte un peu plus cher, mais beaucoup moins que deux camions. Regrouper les lignes dans moins de camions est donc ce qui fait baisser le coût.

### Fonction de vérification des contraintes

In [ ]:
# ---------- Fonction objectif ----------
def cout_camion(poids, supplier_id):
    # Coût d'un camion ; poids inconnu (camion réservé) -> tranche > 4 000 kg
    return cout_transport(POIDS_PAR_DEFAUT if pd.isna(poids) else poids, distances[supplier_id])


def cout_total(planning):
    # Objectif à minimiser : somme des coûts des camions
    return sum(cout_camion(p, s) for p, s in zip(planning["poids"], planning["supplier_id"]))


# ---------- Charge des usines (utilisée par C10 et par le lissage) ----------
def case(date):
    # (n° de semaine dans l'horizon, jour de la semaine) d'une date
    return SEMAINES.index(date.isocalendar().week), date.day_of_week


def charge_depuis_camions(camions, plant):
    # Camions reçus par une usine : une ligne par semaine, une colonne par jour (lundi -> vendredi)
    charge = np.zeros((NB_SEMAINES, 5))
    for d in camions.loc[camions["plant"] == plant, "date"]:
        if d.isocalendar().week in SEMAINES and d.day_of_week < 5:
            charge[case(d)] += 1
    return charge


def charges_camions(camions):
    return {plant: charge_depuis_camions(camions, plant) for plant in USINES}


def ecart(charge_jours):
    # Écart du jour le plus éloigné de la moyenne, en % de la moyenne (0.35 = 35 %)
    charge_jours = np.asarray(charge_jours, dtype=float)
    moyenne = charge_jours.mean()
    return 0.0 if moyenne == 0 else float(np.abs(charge_jours - moyenne).max() / moyenne)


def hors_tolerance(charge_semaines, seuil=None):
    # C10 pour chaque semaine : un jour s'écarte-t-il de la moyenne de plus du seuil (et d'au moins 1 camion) ?
    seuil = SEUIL_LISSAGE if seuil is None else seuil
    moyenne = charge_semaines.mean(axis=1, keepdims=True)
    tolerance = np.maximum(seuil * moyenne, 1.0)
    depasse = (np.abs(charge_semaines - moyenne) > tolerance + 1e-9).any(axis=1)
    return depasse & (charge_semaines.sum(axis=1) >= MIN_CAMIONS_SEMAINE)


# ---------- Vérification des contraintes ----------
# Chaque fonction retourne les cas qui violent la contrainte (table vide = contrainte respectée).
# planning : une ligne par camion (colonnes plant, supplier_id, date, poids, volume, reserve, profil)
# lignes   : une ligne par ligne de besoin (colonnes id_ligne, camion_id, plant, supplier_id, need_date,
#            date_livraison, weight)

def optimises(planning):
    # Camions construits par le moteur (sans les camions réservés de la quarantaine)
    return planning[~planning["reserve"]]


def c1_poids(planning, lignes):
    p = optimises(planning)
    return p[p["poids"] > POIDS_MAX + 1e-6]


def c2_volume(planning, lignes):
    return planning[planning["volume"] > VOLUME_MAX + 1e-6]


def c3_exhaustivite(planning, lignes, ids_attendus):
    # Lignes sans camion, livrées deux fois, ou absentes
    sans_camion = lignes[lignes["camion_id"].isna()].assign(probleme="sans camion")
    doublons = lignes[lignes["id_ligne"].duplicated(keep=False)].assign(probleme="livrée plusieurs fois")
    absentes = pd.DataFrame({"id_ligne": sorted(set(ids_attendus) - set(lignes["id_ligne"])), "probleme": "absente"})
    inconnues = lignes[~lignes["id_ligne"].isin(ids_attendus)].assign(probleme="ligne inconnue")
    return pd.concat([sans_camion, doublons, absentes, inconnues], ignore_index=True)


def c4_un_couple(planning, lignes):
    nb_couples = lignes.groupby("camion_id")[["plant", "supplier_id"]].nunique().max(axis=1)
    return nb_couples[nb_couples > 1].rename("couples").reset_index()


def c5_retard(planning, lignes):
    return lignes[lignes["date_livraison"] > lignes["need_date"]]


def c6_avance(planning, lignes):
    return lignes[lignes["need_date"] - lignes["date_livraison"] > FENETRE]


def c7_jours_ouvres(planning, lignes):
    d = planning["date"]
    return planning[(d.dt.day_of_week >= 5) | (d < DEBUT_LIVRAISONS) | (d > FIN)]   # semaine ajoutée comprise


def c8_jours_fixes(planning, lignes):
    # Profil de 1 à 3 jours, et chaque camion part un de ses jours fixes, sans exception
    p = optimises(planning)
    mauvais_profil = ~p["profil"].map(len).between(1, JOURS_FIXES_MAX)
    hors_profil = np.array([d.day_of_week not in prof for d, prof in zip(p["date"], p["profil"])], dtype=bool)
    return p[mauvais_profil | hors_profil]


MOTIF_POIDS = "besoin d'un jour > 24,5 t"
MOTIF_RESERVE = "ligne sans poids (camion réservé)"


def c9_un_camion_par_jour(planning, lignes):
    # Jours où un couple a 2 camions ou plus, camions réservés compris. 'justifie' = cas impossible à éviter, notifié
    # dans les alertes : besoin d'un seul jour > 24,5 t (2 camions optimisés), ou ligne sans poids (camion réservé)
    par_jour = planning.groupby(["plant", "supplier_id", "date"]).agg(camions=("reserve", "size"), reserves=("reserve", "sum"))
    trop = par_jour[par_jour["camions"] > 1].reset_index()
    besoin_jour = lignes.dropna(subset=["weight"]).groupby(["plant", "supplier_id", "need_date"])["weight"].sum()
    impossibles = set(besoin_jour[besoin_jour > POIDS_MAX].index.droplevel("need_date"))
    lourd = [n - r > 1 and (p, s) in impossibles for p, s, n, r in zip(trop["plant"], trop["supplier_id"], trop["camions"], trop["reserves"])]
    trop["motif"] = [" + ".join(([MOTIF_POIDS] if l else []) + ([MOTIF_RESERVE] if r > 0 else [])) for l, r in zip(lourd, trop["reserves"])]
    trop["justifie"] = [n - r <= 1 or l for n, r, l in zip(trop["camions"], trop["reserves"], lourd)]
    return trop


def c10_lissage(planning, lignes):
    # Semaines hors tolérance, par usine (tous les camions reçus, camions réservés compris)
    lignes_hors = []
    for plant, charge in charges_camions(planning).items():
        hors = hors_tolerance(charge)
        for w in CONTROLE:
            if hors[w]:
                lignes_hors.append({"plant": plant, "semaine": SEMAINES[w], "ecart": ecart(charge[w]),
                                    "camions_par_jour": charge[w].astype(int).tolist()})
    return pd.DataFrame(lignes_hors)


def profil_utilise(dates, profil):
    # True si le couple a au moins une semaine avec un camion sur chacun de ses jours fixes
    # (toujours vrai pour un profil d'un seul jour)
    jours_par_semaine = {}
    for d in dates:
        jours_par_semaine.setdefault(tuple(d.isocalendar())[:2], set()).add(d.day_of_week)
    return any(set(profil) <= jours for jours in jours_par_semaine.values())


def c11_profil_utilise(planning, lignes):
    # Couples à 2 ou 3 jours fixes qui ne livrent jamais tous leurs jours fixes la même semaine
    manques = [(plant, supplier, g["profil"].iloc[0])
               for (plant, supplier), g in optimises(planning).groupby(["plant", "supplier_id"])
               if not profil_utilise(g["date"], g["profil"].iloc[0])]
    return pd.DataFrame(manques, columns=["plant", "supplier_id", "profil"])


CONTRAINTES = [
    ("C1", "Poids ≤ 24,5 t", "dure", c1_poids),
    ("C2", "Volume ≤ 110,4 m³", "dure", c2_volume),
    ("C3", "Chaque ligne livrée une fois, entière", "dure", c3_exhaustivite),
    ("C4", "Un seul couple par camion", "dure", c4_un_couple),
    ("C5", "Jamais de retard", "dure", c5_retard),
    ("C6", "Au plus 7 jours d'avance", "dure", c6_avance),
    ("C7", "Jour ouvré, dans l'horizon", "dure", c7_jours_ouvres),
    ("C8", "Jours fixes (1 à 3 par couple, sans exception)", "dure", c8_jours_fixes),
    ("C9", "Au plus 1 camion par couple et par jour", "dure", c9_un_camion_par_jour),
    ("C10", "Lissage (écart ≤ 35 %)", "souple", c10_lissage),
    ("C11", "Profil utilisé : une semaine sur tous ses jours fixes", "dure", c11_profil_utilise),
]


def alertes(planning, lignes):
    # Cas où une règle ne peut pas être respectée, à notifier : 2 camions ou plus un même jour pour un couple (C9),
    # camion réservé hors des jours fixes du couple (C8) ; avec la cause et l'action à mener
    a_faire = {MOTIF_POIDS: "aucune : le besoin du jour dépasse la capacité d'un camion, 2 camions sont nécessaires",
               MOTIF_RESERVE: "obtenir le poids de la ligne auprès du client pour pouvoir la consolider"}
    cas = []
    c9 = c9_un_camion_par_jour(planning, lignes)
    for r in c9[c9["justifie"]].itertuples():
        cas.append({"regle": "C9 : 2 camions ou plus le même jour", "usine": r.plant, "fournisseur": r.supplier_id,
                    "date": r.date, "camions": int(r.camions), "cause": r.motif,
                    "a_faire": " ; ".join(a_faire[m] for m in r.motif.split(" + "))})
    profils = optimises(planning).groupby(["plant", "supplier_id"])["profil"].first()
    for r in planning[planning["reserve"]].itertuples():
        profil = profils.get((r.plant, r.supplier_id))
        if profil is None or r.date.day_of_week not in profil:
            cas.append({"regle": "C8 : camion hors jours fixes", "usine": r.plant, "fournisseur": r.supplier_id,
                        "date": r.date, "camions": 1,
                        "cause": MOTIF_RESERVE + ("" if profil is not None else ", couple sans autre livraison (pas de jours fixes)"),
                        "a_faire": a_faire[MOTIF_RESERVE]})
    colonnes = ["regle", "usine", "fournisseur", "date", "camions", "cause", "a_faire"]
    return pd.DataFrame(cas, columns=colonnes).sort_values(["regle", "usine", "date", "fournisseur"]).reset_index(drop=True)


def verifier_contraintes(planning, lignes, ids_attendus):
    # Tableau de synthèse : nombre de violations de chaque contrainte
    resultats = []
    for code_c, nom, type_c, fonction in CONTRAINTES:
        args = (planning, lignes, ids_attendus) if fonction is c3_exhaustivite else (planning, lignes)
        violations = fonction(*args)
        remarque, notifies = "", 0
        if fonction is c8_jours_fixes:
            notifies = int(alertes(planning, lignes)["regle"].str.startswith("C8").sum())
            remarque = f"{notifies} camions réservés hors jours fixes, notifiés (alertes)"
        if fonction is c9_un_camion_par_jour:
            justifies = violations[violations["justifie"]]
            notifies = len(justifies)
            remarque = (f"{notifies} cas impossibles, notifiés (alertes) : {int(justifies['motif'].str.contains('24,5').sum())} "
                        f"besoin d'un jour > 24,5 t, {int(justifies['motif'].str.contains('sans poids').sum())} ligne sans poids")
            violations = violations[~violations["justifie"]]
        if fonction is c10_lissage:
            remarque = f"sur {len(USINES) * len(CONTROLE)} semaines contrôlées"
        nb = len(violations)
        statut = "respectée" if nb == 0 else ("violée" if type_c == "dure" else f"{nb} semaine(s) hors tolérance")
        if nb == 0 and notifies:
            statut = "respectée, sauf cas impossibles notifiés"
        resultats.append({"Contrainte": code_c, "Description": nom, "Type": type_c, "Violations": nb,
                          "Statut": statut, "Remarque": remarque})
    return pd.DataFrame(resultats).set_index("Contrainte")

## 2. Choix de l'algorithme

Chaque membre du groupe a conçu son propre algorithme, sur les mêmes données et les mêmes contraintes. On les a comparés sur les mêmes critères, puis on a retenu celui qui donnait le meilleur résultat. Deux algorithmes sont ressortis : un algorithme mathématique, que nous n'avons pas retenu, et un algorithme à approche mécanique, que nous avons choisi.

### Tentative d'algorithme mathématique

Un algorithme mathématique a été testé. Son principe :

Pour chaque couple fournisseur × usine, on cherche les jours de livraison fixes qui coûtent le moins cher, puis on répartit ces jours entre les couples d'une même usine pour éviter les pics.

D'abord, il calcule le k de chaque couple : le nombre minimum de camions dont il a besoin sur une semaine, k = ⌈max(poids / 24,5 t ; volume / 110,4 m³)⌉ sur la charge hebdomadaire. Il retient le k de la pire semaine, pour qu'une même fréquence tienne toutes les semaines. La fréquence en est déduite, la plus basse possible parmi 1, 2, 3 ou 5 livraisons par semaine : c'est le plus petit élément de cette liste qui est ≥ k_max. Ici, 177 couples sur 180 ont k_max = 1, donc une seule livraison par semaine leur suffit.

Ensuite, il teste toutes les combinaisons de jours à cette fréquence, livre chaque besoin au dernier jour fixe qui le précède (jamais en retard, 7 jours d'avance au plus) et garde le profil le moins cher.

Enfin, un lissage par usine est effectué. Pour mesurer une répartition, on additionne les carrés du nombre de camions de chaque jour. Cette somme est plus petite quand les jours ont la même charge, et elle pénalise fortement les pics : par exemple, 10 camions répartis à 2 par jour sur 5 jours donnent 20, alors que 10 camions le même jour donnent 100. Un mouvement n'est accepté que s'il fait baisser cette somme, ce qui rend le lissage facile à mesurer et garantit que l'algorithme s'arrête. Le résultat passe de 1 741 à 869 camions et de 177 321 € à 124 107 € (-30,0 %), avec un pic de 24 camions par usine et par jour au lieu de 54 dans la baseline.

L'algorithme ne respecte les contraintes que si l'on ne compte pas les lignes sans poids (non optimisables). Il reste aussi moins performant que l'algorithme mécanique : 869 camions et 124 107 €, contre 787 camions et 116 118 €. Du fait de ces performances, cet algorithme n'a pas été retenu. Le code n'a pas été ajouté pour ne pas surcharger le notebook du milestone 3.


### Comparaison des deux approches

| | Algorithme mécanique (règles) | Algorithme mathématique |
|---|---|---|
| Principe | consolidation par couple avec une fenêtre de 7 jours ; chaque couple teste tous ses profils de jours fixes et garde le moins cher ; lissage par changements de profil au moindre surcoût ; report des lignes sur le jour fixe suivant | fréquence de livraison déduite du nombre minimum de camions par semaine (k) ; chaque couple teste les combinaisons de jours à cette fréquence et garde la moins chère ; lissage par usine en faisant baisser la somme des carrés des camions par jour |
| Coût total (camions réservés compris) | 116 118 € | 124 107 € |
| Camions | 787 | 869 |
| Économie vs baseline | 61 203 € (-34,5 %) | 53 214 € (-30,0 %) |
| Semaines hors tolérance | 2 / 20 | non mesuré |

Algorithme retenu : l'algorithme mécanique.

Justification : il coûte 7 989 € de moins et utilise 82 camions de moins que l'algorithme mathématique, et il respecte toutes les contraintes dures, y compris avec les lignes sans poids.

La suite du notebook présente l'algorithme mécanique.

## 3. Fonctionnement global

Le moteur prend en entrée le planning des besoins (une ligne = un matériau à livrer à une usine un jour donné) et produit le planning des camions. Il enchaîne quatre blocs :

```
Données préparées aux milestones 1 et 2 (besoins + lignes de la quarantaine)
        │
        ▼
Bloc 1 : jours fixes
  chaque couple teste ses 25 profils (1, 2 ou 3 jours), sans jamais 2 camions le même jour et en
  utilisant vraiment tous ses jours fixes (C11), et garde le moins cher ; le camion part le dernier jour fixe avant son premier besoin
  et prend les besoins des 7 jours suivants (24,5 t et 110,4 m³ max)
        │
        ▼
Bloc 2 : contrôle de l'équilibre
  pour chaque usine et chaque semaine : un jour s'écarte-t-il de plus de 35 % de la moyenne ?
        │ non ──► rien à faire
        │ oui
        ▼
Bloc 3 : lissage
  on change le profil (les jours fixes) de certains couples de l'usine, au moindre surcoût,
  jusqu'à repasser dans la tolérance
        │
        ▼
Bloc 4 : report et planning final
  les lignes qui peuvent attendre partent avec le camion du jour fixe suivant (s'il a la place
  et que ça ne coûte rien), puis on construit le planning complet
        │
        ▼
Validation des contraintes  ──►  Sortie 1 : profil de chaque couple
                                 Sortie 2 : contenu de chaque camion
```

Pourquoi cet ordre : on optimise d'abord le coût (bloc 1), car c'est l'objectif. Le lissage (bloc 3) ne fait ensuite que les changements nécessaires, là où une usine est déséquilibrée, en payant le moins possible. Le report (bloc 4) vient en dernier : il ne change ni les jours fixes ni le nombre de camions, il rapproche seulement des lignes de leur besoin.

| Paramètre | Valeur | Rôle |
|---|---|---|
| `POIDS_MAX`, `VOLUME_MAX` | 24 500 kg, 110,4 m³ | capacité d'un camion |
| `FENETRE` | 7 jours | avance maximale d'une livraison |
| `JOURS_FIXES_MAX` | 3 | nombre maximum de jours fixes par couple |
| `SEUIL_LISSAGE` | 35 % | écart toléré entre un jour et la moyenne de la semaine |
| `MIN_CAMIONS_SEMAINE` | 5 | en dessous, une semaine n'est pas contrôlée |

## 4. Moteur, bloc par bloc

On part des données préparées aux milestones 1 et 2 : les besoins nettoyés (silver), les lignes de la quarantaine et la baseline. La cellule ci-dessous les charge, crée un camion réservé pour chaque ligne de la quarantaine sans poids (comme dans la baseline du milestone 2), et fixe l'horizon du planning (semaines et usines). L'horizon commence une semaine avant le premier besoin : la semaine S45, ajoutée pour que les premiers besoins puissent partir un jour fixe.

In [ ]:
# ---------- Données des milestones 1 et 2 ----------
silver = pd.read_csv(SILVER_CLEAN_PATH, parse_dates=["need_date"])
quarantaine = pd.read_csv(QUARANTAINE_PATH, parse_dates=["need_date"])
N_SILVER = len(silver)
sans_poids = quarantaine["weight"].isna()
a_consolider = ~sans_poids & (quarantaine["weight"] <= POIDS_MAX)   # la ligne de plus de 24,5 t est exclue

# Besoins à consolider : silver + lignes de la quarantaine dont le poids est connu
besoins = (pd.concat([silver, quarantaine.loc[a_consolider, silver.columns]], ignore_index=True)
           .sort_values(["plant", "supplier_id", "need_date", "weight"], ascending=[True, True, True, False])
           [["need_date", "supplier_id", "material", "plant", "unloading_point", "schedule_quantity", "weight", "volume"]])

# Lignes sans poids : un camion réservé par ligne, le jour du besoin
lignes_reserves = quarantaine[sans_poids].drop(columns="flags").reset_index(drop=True)
lignes_reserves["numero"] = range(1, len(lignes_reserves) + 1)
lignes_reserves["id_ligne"] = "Q" + lignes_reserves["numero"].astype(str)
lignes_reserves["date_livraison"], lignes_reserves["avance_jours"] = lignes_reserves["need_date"], 0
camions_reserves = pd.DataFrame({
    "numero": lignes_reserves["numero"], "plant": lignes_reserves["plant"], "supplier_id": lignes_reserves["supplier_id"],
    "date": lignes_reserves["need_date"], "nb_lignes": 1, "poids": np.nan,
    "volume": lignes_reserves["volume"], "cout": [cout_camion(np.nan, s) for s in lignes_reserves["supplier_id"]],
    "motif": "poids inconnu"})
COUT_RESERVE, NB_RESERVE = camions_reserves["cout"].sum(), len(camions_reserves)


def avec_reserves(camions):
    return pd.concat([camions, camions_reserves.drop(columns="numero")], ignore_index=True)


# Baseline du milestone 2 (sans ses camions réservés, comptés à part : les mêmes pour toutes les versions)
camions_base = pd.read_csv(BASELINE_PATH, parse_dates=["date"])
assert camions_base["reserve"].sum() == NB_RESERVE
camions_base = camions_base[~camions_base["reserve"]].copy()
camions_base["cout"] = [cout_camion(p, s) for p, s in zip(camions_base["poids"], camions_base["supplier_id"])]
COUT_BASELINE = camions_base["cout"].sum()

# Horizon du planning
DEBUT, FIN = besoins["need_date"].min(), besoins["need_date"].max()   # lundi 09/11 -> vendredi 18/12
DEBUT_LIVRAISONS = DEBUT - pd.Timedelta(days=7)                        # lundi 02/11 : semaine ajoutée avant le planning
SEMAINES = sorted({int(w) for w in besoins["need_date"].dt.isocalendar().week} | {int(DEBUT_LIVRAISONS.isocalendar().week)})
NB_SEMAINES = len(SEMAINES)
CONTROLE = list(range(1, NB_SEMAINES - 1))   # semaine ajoutée et dernière semaine, incomplètes : pas contrôlées
USINES = sorted(besoins["plant"].unique())
IDS_ATTENDUS = {f"L{i}" for i in besoins.index} | set(lignes_reserves["id_ligne"])

print(f"Besoins à consolider : {len(besoins)} lignes, {besoins.groupby(['plant', 'supplier_id']).ngroups} couples, "
      f"{len(USINES)} usines, du {DEBUT:%d/%m} au {FIN:%d/%m} (livraisons possibles dès le {DEBUT_LIVRAISONS:%d/%m})")
print(f"Camions réservés (lignes sans poids) : {NB_RESERVE}, {eur(COUT_RESERVE)}")
print(f"Baseline : {len(camions_base)} camions, {eur(COUT_BASELINE)} (+ les camions réservés)")

### Bloc 1 : jours fixes

Rôle : donner à chaque couple ses jours fixes de livraison (1 à 3 jours par semaine), en regroupant ses lignes dans le moins de camions possible.

Principe : pour un profil donné, la consolidation suit la même boucle que l'algorithme de base, avec une seule différence. Quand un camion s'ouvre, il ne part pas le jour du plus ancien besoin restant, mais le dernier jour fixe du couple avant ce besoin (ou le jour même). Il prend ensuite les besoins jusqu'à sa date + 7 jours, du plus ancien au plus récent, tant qu'il reste de la place (24,5 t, 110,4 m³).

```
Couple au profil « mardi + vendredi »
besoin restant le plus ancien : jeudi 19/11     ──►  camion le mardi 17/11 (prend les besoins du 17/11 au 24/11)
besoin restant le plus ancien : mercredi 25/11  ──►  camion le mardi 24/11 (prend les besoins du 24/11 au 01/12)
besoin restant le plus ancien : samedi 05/12    ──►  camion le vendredi 04/12 ...
```

Choix du profil : chaque couple teste ses 25 profils (5 à 1 jour, 10 à 2 jours, 10 à 3 jours). On ne garde que ceux qu'il utilise vraiment, c'est-à-dire où il livre au moins une semaine sur tous ses jours fixes (C11), et où il n'a jamais 2 camions le même jour (C9), fonction `autorises`, puis on prend le moins cher (à coût égal : le moins de camions, puis le moins de jours). Si aucun profil ne le permet, parce que le besoin d'un seul jour dépasse 24,5 t, le couple garde tous ses profils qui respectent C11 et prend le moins cher.

Début du planning : on peut livrer dans la semaine qui précède le planning (S45). Avec le profil « jeudi », un besoin du mardi 10/11 part le jeudi 05/11, comme n'importe quel autre besoin : le dernier jour fixe avant un besoin est toujours à moins de 7 jours, donc tous les camions partent un jour fixe, dès la première semaine.

Profil utilisé (C11) : un couple au profil « mardi + vendredi » doit avoir au moins une semaine avec un camion le mardi et un camion le vendredi. Comme un camion prend les besoins des 7 jours suivants, la plupart des couples n'ont besoin que d'un camion par semaine : leurs profils à 2 ou 3 jours ne livrent alors jamais tous leurs jours la même semaine, et sont exclus. Les profils d'un seul jour sont toujours utilisés, donc chaque couple garde au moins un profil.

Trois variantes sont construites pour mesurer l'intérêt d'un deuxième et d'un troisième jour : 1 jour fixe seulement, 1 ou 2 jours fixes, et 1 à 3 jours fixes. Elles sont comparées à la référence sans jours fixes (l'algorithme de base du milestone 2), qui revient à autoriser les livraisons les 5 jours de la semaine.

In [ ]:
PROFILS = [p for n in range(1, JOURS_FIXES_MAX + 1) for p in itertools.combinations(range(5), n)]   # 5 + 10 + 10 = 25 profils
nom_profil = lambda profil: " + ".join(JOURS[j] for j in profil)


def date_camion(besoin, profil):
    # Dernier jour du profil avant ou égal au besoin (à moins de 7 jours ; dans la semaine ajoutée pour les premiers besoins)
    return max(besoin - pd.Timedelta(days=(besoin.day_of_week - j) % 7) for j in profil)


def consolider_couple(lignes, profil):
    # Consolidation d'un couple avec des jours fixes (lignes triées par date, puis poids décroissant)
    restantes = list(lignes.itertuples())
    camions = []
    while restantes:
        date = date_camion(restantes[0].need_date, profil)
        poids, volume, contenu, i = 0.0, 0.0, [], 0
        while i < len(restantes):
            ligne = restantes[i]
            if ligne.need_date > date + FENETRE:
                break                                   # hors fenêtre de 7 jours : camion fermé
            if poids + ligne.weight > POIDS_MAX or volume + ligne.volume > VOLUME_MAX:
                i += 1                                  # ne rentre pas : ligne suivante
                continue
            poids, volume = poids + ligne.weight, volume + ligne.volume
            contenu.append(ligne.Index)
            restantes.pop(i)
            i = 0
        camions.append({"date": date, "poids": poids, "volume": volume, "lignes": contenu})
    return pd.DataFrame(camions)


# Toutes les options : chaque couple × chaque profil
scenarios = {}
for (plant, supplier), lignes_couple in besoins.groupby(["plant", "supplier_id"]):
    for profil in PROFILS:
        camions = consolider_couple(lignes_couple, profil)
        camions["cout"] = [cout_camion(p, supplier) for p in camions["poids"]]
        scenarios[(plant, supplier, profil)] = camions
# Référence : l'algorithme de base du milestone 2 (sans jours fixes), où le camion part le jour du plus ancien
# besoin, revient à livrer les 5 jours de la semaine
REFERENCE = (0, 1, 2, 3, 4)
camions_algo = pd.concat([consolider_couple(lignes_couple, REFERENCE).assign(plant=plant, supplier_id=supplier)
                          for (plant, supplier), lignes_couple in besoins.groupby(["plant", "supplier_id"])], ignore_index=True)
camions_algo["profil"] = [REFERENCE] * len(camions_algo)
camions_algo["cout"] = [cout_camion(p, s) for p, s in zip(camions_algo["poids"], camions_algo["supplier_id"])]
COUT_ALGO = camions_algo["cout"].sum()
print(f"Référence sans jours fixes : {len(camions_algo)} camions, {eur(COUT_ALGO)}")

cout_option = {cle: c["cout"].sum() for cle, c in scenarios.items()}
nb_camions_option = {cle: len(c) for cle, c in scenarios.items()}
charge_option = {(p, s, prof): charge_depuis_camions(c.assign(plant=p), p) for (p, s, prof), c in scenarios.items()}
max_jour = {cle: int(c.groupby("date").size().max()) for cle, c in scenarios.items()}   # camions max un même jour
utilise = {(p, s, prof): profil_utilise(c["date"], prof) for (p, s, prof), c in scenarios.items()}   # C11


def autorises(plant, supplier, profils_variante):
    # Profils de la variante que le couple utilise vraiment (C11 : une semaine au moins sur tous ses jours fixes)
    # et qui ne lui mettent jamais 2 camions le même jour (C9).
    # Si aucun ne respecte C9 (besoin d'un seul jour > 24,5 t) : tous les profils utilisés de la variante
    utilises = [p for p in profils_variante if utilise[(plant, supplier, p)]]
    assert utilises, f"aucun profil utilisé pour {plant} / {supplier}"
    ok = [p for p in utilises if max_jour[(plant, supplier, p)] <= 1]
    return ok if ok else utilises


def choix_moins_cher(profils_variante):
    # Profil le moins cher de chaque couple (à égalité : moins de camions, puis moins de jours)
    choix = {}
    for cle in sorted((k for k in scenarios if k[2] in autorises(k[0], k[1], profils_variante)),
                      key=lambda k: (k[0], k[1], round(cout_option[k], 2), nb_camions_option[k], len(k[2]), k[2])):
        choix.setdefault(cle[:2], cle[2])
    return choix


cout_choix = lambda choix: sum(cout_option[(p, s, prof)] for (p, s), prof in choix.items())
VARIANTES = {"1 jour fixe": [p for p in PROFILS if len(p) == 1], "1 ou 2 jours fixes": [p for p in PROFILS if len(p) <= 2],
             "1 à 3 jours fixes": PROFILS}
choix_initiaux = {nom: choix_moins_cher(profils) for nom, profils in VARIANTES.items()}

impossibles = sorted({k[:2] for k in scenarios} - {k[:2] for k in scenarios if max_jour[k] <= 1})
multi = [k for k in scenarios if len(k[2]) > 1]
print(f"{len(scenarios)} options évaluées ({len(choix_initiaux['1 à 3 jours fixes'])} couples × {len(PROFILS)} profils) ; "
      f"options à 2 ou 3 jours utilisées en entier au moins une semaine (C11) : {sum(utilise[k] for k in multi)} sur {len(multi)}")
print(f"Couples qui ne peuvent pas avoir 1 seul camion par jour (besoin d'un jour > 24,5 t) : {impossibles}")
for nom, choix in choix_initiaux.items():
    nb = sum(nb_camions_option[(p, s, prof)] for (p, s), prof in choix.items())
    nb_jours = pd.Series([len(prof) for prof in choix.values()]).value_counts()
    print(f"{nom:<20}: {nb} camions, {eur(cout_choix(choix))} (+{eur(cout_choix(choix) - COUT_ALGO)} vs algorithme de base), "
          f"couples à 1 / 2 / 3 jours : {nb_jours.get(1, 0)} / {nb_jours.get(2, 0)} / {nb_jours.get(3, 0)}")

Résultat (hors camions réservés, identiques dans toutes les versions) :

| Variante | Camions | Coût | Surcoût vs algorithme de base | Couples à 1 / 2 / 3 jours |
|---|---|---|---|---|
| 1 jour fixe | 632 | 81 580 € | +5 573 € | 173 / 0 / 0 |
| 1 ou 2 jours fixes | 632 | 81 580 € | +5 573 € | 172 / 1 / 0 |
| 1 à 3 jours fixes | 632 | 81 580 € | +5 573 € | 172 / 1 / 0 |

Les trois variantes donnent presque le même résultat, à cause de la règle du profil utilisé (C11) : seules 14 options à 2 ou 3 jours sur 3 460 livrent au moins une semaine sur tous leurs jours fixes. Un camion prend les besoins des 7 jours suivants, donc presque tous les couples n'ont besoin que d'un camion par semaine : avec un profil à 2 jours, ils livreraient tantôt l'un, tantôt l'autre, jamais les deux la même semaine. Un seul couple garde 2 jours fixes : SUP_00117 → PLT_004, qui reçoit environ 45 t par semaine et livre le lundi et le mercredi.

Le surcoût des jours fixes (+5 573 € par rapport à l'algorithme de base, où chaque camion part le jour exact du premier besoin) vient donc du passage à un seul jour de livraison par semaine pour presque tous les couples.

Seuls 2 couples ne peuvent pas avoir un seul camion par jour : SUP_00037 → PLT_002 et SUP_00046 → PLT_004, dont le besoin d'un même jour dépasse 24,5 t.

### Bloc 2 : contrôle de l'équilibre

Rôle : repérer les usines qui reçoivent trop de camions certains jours et pas assez d'autres.

Principe : pour chaque usine et chaque semaine, on compte les camions reçus chaque jour (C10). Une semaine est hors tolérance si un jour s'écarte de la moyenne de la semaine de plus de 35 %, avec au moins 1 camion de marge (le choix de 35 % est expliqué à la fin du bloc 3) : une usine qui reçoit 5 camions dans la semaine ne peut pas faire mieux qu'un par jour. Les semaines de moins de 5 camions ne sont pas contrôlées, ni la semaine ajoutée avant le planning (S45), qui ne reçoit que quelques livraisons anticipées et jamais le lundi, ni la dernière semaine (S51), incomplète. Les camions réservés de la quarantaine sont comptés, car ils arrivent aux quais.

In [ ]:
CHARGE_RESERVEE = charges_camions(camions_reserves)   # charge fixe, que le lissage ne déplace pas


def charges_choix(choix, fond=None):
    # Charge de chaque usine pour un choix de profils, avec les camions réservés
    fond = CHARGE_RESERVEE if fond is None else fond
    return {plant: fond[plant] + sum(charge_option[(p, s, prof)] for (p, s), prof in choix.items() if p == plant)
            for plant in USINES}


def tableau_controle(charges):
    # Par usine : camions moyens par jour, pire semaine et décision de lissage
    lignes = []
    for plant, charge in charges.items():
        hors = hors_tolerance(charge[CONTROLE])
        lignes.append({"plant": plant, **dict(zip(JOURS, charge.mean(axis=0))),
                       "Pire semaine": max(ecart(charge[w]) for w in CONTROLE),
                       "Semaines hors tolérance": int(hors.sum()), "Lissage nécessaire": bool(hors.any())})
    return pd.DataFrame(lignes).set_index("plant")


def heatmap(charges_par_etape, titre):
    # Camions reçus par semaine × jour pour chaque usine ; une ligne de graphiques par étape
    fig, axes = plt.subplots(len(charges_par_etape), len(USINES), figsize=(16, 3.8 * len(charges_par_etape)),
                             sharex=True, sharey=True, squeeze=False)
    for ligne, (etape, charges) in enumerate(charges_par_etape.items()):
        for col, plant in enumerate(USINES):
            ax, charge = axes[ligne, col], charges[plant]
            hors = hors_tolerance(charge)
            im = ax.imshow(charge, cmap="YlGnBu", vmin=0, vmax=20, aspect="auto")
            for w in range(NB_SEMAINES):
                for j in range(5):
                    v = int(charge[w, j])
                    ax.text(j, w, v, ha="center", va="center", color="white" if v > 12 else "black",
                            fontweight="bold" if hors[w] and w in CONTROLE else "normal")
            ax.set_xticks(range(5), [j[:3] for j in JOURS])
            ax.set_yticks(range(NB_SEMAINES), [f"S{s}" for s in SEMAINES])
            ax.set_title(f"{plant} {etape}\n{int(hors[CONTROLE].sum())} semaine(s) hors tolérance", fontsize=10)
    fig.colorbar(im, ax=axes, label="Camions reçus", shrink=0.8)
    fig.suptitle(f"{titre}\n(en gras : semaine hors tolérance ; S{SEMAINES[0]} et S{SEMAINES[-1]} non contrôlées)", fontsize=13, y=1.02)
    plt.show()


format_controle = {**{j: "{:.1f}" for j in JOURS}, "Pire semaine": pourcent}
for nom, choix in choix_initiaux.items():
    print(f"{nom:<20}: {tableau_controle(charges_choix(choix))['Semaines hors tolérance'].sum()} semaines hors tolérance "
          f"sur {len(USINES) * len(CONTROLE)}")
tableau_controle(charges_choix(choix_initiaux["1 à 3 jours fixes"])).style.format(format_controle)

Résultat : avant lissage, les 20 semaines contrôlées sont hors tolérance, dans les trois variantes. Chaque couple a pris son jour le moins cher, sans regarder les autres couples de l'usine : beaucoup tombent sur les mêmes jours. Les 4 usines doivent être lissées.

### Bloc 3 : lissage

Rôle : ramener chaque semaine dans la tolérance, au moindre surcoût.

Principe : on ne déplace pas des camions un par un, on change les jours fixes d'un couple (son profil), pour que ses livraisons restent régulières. Pour chaque usine hors tolérance, on répète :
1. tester tous les changements de profil possibles d'un couple de l'usine (autre jour, autre paire ou trio de jours), parmi les profils autorisés (profil utilisé, jamais 2 camions le même jour) 

2. garder ceux qui rendent les semaines plus régulières, c'est-à-dire qui font baisser la dispersion : la somme, sur les semaines contrôlées, des carrés des écarts à la moyenne de la semaine 

3. appliquer celui qui coûte le moins par unité d'amélioration (les changements gratuits d'abord) 

jusqu'à ce qu'aucune semaine ne soit hors tolérance, ou qu'aucun changement n'améliore plus rien. Les usines déjà équilibrées ne sont pas touchées.


In [ ]:
def dispersion(charge):
    # Somme, sur les semaines contrôlées, des carrés des écarts à la moyenne de la semaine
    charge = charge[CONTROLE]
    return float(((charge - charge.mean(axis=1, keepdims=True)) ** 2).sum())


def lisser_par_couple(choix, plant, profils_variante, seuil=None, fond=None):
    # Change le profil de couples de `plant` jusqu'à ce qu'aucune semaine contrôlée ne soit hors tolérance
    choix = dict(choix)
    charge = charges_choix(choix, fond)[plant]
    journal = []
    while hors_tolerance(charge[CONTROLE], seuil).any():
        dispersion_actuelle, meilleur = dispersion(charge), None
        for (p, s), prof in choix.items():
            if p != plant:
                continue
            for prof2 in autorises(p, s, profils_variante):     # jamais 2 camions le même jour
                if prof2 == prof:
                    continue
                nouvelle = charge - charge_option[(p, s, prof)] + charge_option[(p, s, prof2)]
                gain = dispersion_actuelle - dispersion(nouvelle)
                if gain <= 1e-9:
                    continue
                surcout = cout_option[(p, s, prof2)] - cout_option[(p, s, prof)]
                cle = (max(surcout, 0) / gain, -gain)
                if meilleur is None or cle < meilleur["cle"]:
                    meilleur = {"cle": cle, "supplier_id": s, "de": prof, "vers": prof2, "surcout": surcout,
                                "charge": nouvelle}
        if meilleur is None:
            break                                               # plus aucun changement n'améliore les semaines
        choix[(plant, meilleur["supplier_id"])] = meilleur["vers"]
        charge = meilleur["charge"]
        journal.append({"plant": plant, "supplier_id": meilleur["supplier_id"], "de": nom_profil(meilleur["de"]),
                        "vers": nom_profil(meilleur["vers"]), "surcout": round(meilleur["surcout"], 2),
                        "semaines_hors_tolerance_apres": int(hors_tolerance(charge[CONTROLE], seuil).sum())})
    return choix, journal


def lisser(choix, profils_variante, seuil=None, fond=None):
    # Lisse toutes les usines qui ont au moins une semaine hors tolérance
    choix, journal = dict(choix), []
    for plant in USINES:
        if hors_tolerance(charges_choix(choix, fond)[plant][CONTROLE], seuil).any():
            choix, journal_u = lisser_par_couple(choix, plant, profils_variante, seuil, fond)
            journal += journal_u
    return choix, pd.DataFrame(journal)


choix_lisses, journaux = {}, {}
for nom, profils in VARIANTES.items():
    choix_lisses[nom], journaux[nom] = lisser(choix_initiaux[nom], profils)
    avant = tableau_controle(charges_choix(choix_initiaux[nom]))["Semaines hors tolérance"].sum()
    apres = tableau_controle(charges_choix(choix_lisses[nom]))["Semaines hors tolérance"].sum()
    print(f"{nom:<20}: {len(journaux[nom])} changements de profil, "
          f"surcoût {eur(cout_choix(choix_lisses[nom]) - cout_choix(choix_initiaux[nom]))} -> {eur(cout_choix(choix_lisses[nom]))} ; "
          f"semaines hors tolérance {avant} -> {apres} (sur {len(USINES) * len(CONTROLE)})")

def doublons_evitables(choix):
    # Couples avec 2 camions un même jour alors qu'un autre profil utilisé l'éviterait (C9)
    return [(p, s) for (p, s), prof in choix.items()
            if max_jour[(p, s, prof)] > 1 and max_jour[(p, s, autorises(p, s, PROFILS)[0])] <= 1]


# Variante retenue : la moins chère après lissage, parmi celles qui respectent C9
for nom, choix in choix_lisses.items():
    if doublons_evitables(choix):
        print(f"{nom} écartée : 2 camions un même jour évitables pour {doublons_evitables(choix)}")
PRINCIPALE = min((nom for nom in VARIANTES if not doublons_evitables(choix_lisses[nom])),
                 key=lambda nom: cout_choix(choix_lisses[nom]))
print(f"\nVariante retenue : {PRINCIPALE}")
tableau_controle(charges_choix(choix_lisses[PRINCIPALE])).style.format(format_controle)

In [ ]:
# Journal des changements de profil de la variante retenue (traçabilité)
journaux[PRINCIPALE]

#### Choix du seuil de lissage (35 %)

La question : quel seuil de tolérance choisir pour le lissage ? Plus le seuil est bas, plus on lisse : les semaines sont plus régulières, mais des couples sont éloignés de leur profil le moins cher, donc le coût monte.

La méthode : on refait le lissage pour plusieurs seuils, du « sans lissage » jusqu'à 15 %, et on compare trois indicateurs :

| Indicateur | Ce qu'il mesure | On veut |
|---|---|---|
| Surcoût du lissage | ce que le lissage ajoute au coût de transport, par rapport à « sans lissage » | le plus bas possible |
| Jour le plus chargé | le plus grand nombre de camions reçus un même jour par une usine, sur toute la période | le plus bas possible |
| Irrégularité moyenne | en moyenne, de combien le jour le plus éloigné de la semaine s'écarte de la moyenne de cette semaine (ex. 50 % : avec 6 camions par jour en moyenne, un jour en reçoit 9 ou 3) | le plus bas possible |

Les deux derniers indicateurs sont calculés de la même façon pour tous les seuils, sur les semaines contrôlées, camions réservés compris : on peut donc comparer les seuils entre eux.

In [ ]:
SEUILS_TESTES = [None, 1.0, 0.8, 0.6, 0.5, 0.4, 0.35, 0.3, 0.25, 0.2, 0.15]


def indicateurs_equilibre(charges):
    # Jour le plus chargé et irrégularité moyenne (pondérée par les camions de chaque semaine),
    # sur les semaines contrôlées
    ecarts, poids, pic = [], [], 0
    for charge in charges.values():
        for w in CONTROLE:
            if charge[w].sum() > 0:
                ecarts.append(ecart(charge[w]))
                poids.append(charge[w].sum())
                pic = max(pic, charge[w].max())
    return int(pic), np.average(ecarts, weights=poids)


etude = []
for seuil in SEUILS_TESTES:
    choix = choix_initiaux[PRINCIPALE] if seuil is None else lisser(choix_initiaux[PRINCIPALE], VARIANTES[PRINCIPALE], seuil)[0]
    pic, irregularite = indicateurs_equilibre(charges_choix(choix))   # camions réservés compris
    etude.append({"Seuil": "sans lissage" if seuil is None else pourcent(seuil).replace(",0", ""), "seuil": seuil,
                  "Coût total": cout_choix(choix) + COUT_RESERVE,
                  "Camions": sum(nb_camions_option[(p, s, prof)] for (p, s), prof in choix.items()) + NB_RESERVE,
                  "Jour le plus chargé (camions)": pic, "Irrégularité moyenne": irregularite})
etude = pd.DataFrame(etude)
etude.insert(3, "Surcoût du lissage", etude["Coût total"] - etude.loc[0, "Coût total"])

ligne_retenue = etude.index[etude["seuil"] == SEUIL_LISSAGE][0]
(etude.drop(columns="seuil").style
 .format({"Coût total": eur, "Surcoût du lissage": eur, "Irrégularité moyenne": pourcent})
 .apply(lambda l: ["background-color: #fde9c9" if l.name == ligne_retenue else ""] * len(l), axis=1)
 .hide(axis="index"))

In [ ]:
# ---------- Trois indicateurs selon le seuil (de gauche à droite : on lisse de plus en plus) ----------
x = np.arange(len(etude))
graphes = [("Surcoût du lissage", "Ce que coûte le lissage (€)", "royalblue", eur),
           ("Jour le plus chargé (camions)", "Jour le plus chargé (camions)", "darkorange", lambda v: f"{v:.0f}"),
           ("Irrégularité moyenne", "Irrégularité moyenne des semaines", "seagreen", pourcent)]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), sharex=True)
for ax, (colonne, titre, couleur, fmt) in zip(axes, graphes):
    valeurs = etude[colonne] * (100 if colonne == "Irrégularité moyenne" else 1)
    ax.plot(x, valeurs, marker="o", color=couleur)
    for xi, (v, brut) in enumerate(zip(valeurs, etude[colonne])):
        if xi == 0 or brut != etude[colonne].iloc[xi - 1]:          # pas d'étiquette répétée
            ax.annotate(fmt(brut), (xi, v), textcoords="offset points", xytext=(0, 7), ha="center", fontsize=8)
    ax.axvline(ligne_retenue, color="red", linestyle="--", alpha=0.6)
    ax.text(ligne_retenue, ax.get_ylim()[1], " seuil retenu", color="red", va="top", fontsize=9)
    ax.set_title(titre)
    ax.set_xticks(x, etude["Seuil"], rotation=45, ha="right")
    ax.set_xlabel("Seuil (à droite : on lisse davantage)")
    ax.margins(y=0.15)
    ax.grid(alpha=0.3)
axes[2].yaxis.set_major_formatter(lambda v, _: f"{v:.0f} %")
fig.suptitle("Plus on baisse le seuil, plus on lisse : ce que ça coûte et ce que ça apporte", fontsize=13)
plt.tight_layout()
plt.show()

#### Analyse : choix du seuil

Comment lire les graphiques : de gauche à droite, on baisse le seuil, donc on lisse de plus en plus. Le coût monte (à gauche), tandis que le jour le plus chargé et l'irrégularité baissent (au milieu et à droite).

| Seuil | Coût total | Surcoût du lissage | Jour le plus chargé | Irrégularité moyenne |
|---|---|---|---|---|
| sans lissage | 115 086 € | 0 € | 33 camions | 120 % |
| 100 % | 116 091 € | 1 005 € | 17 camions | 55 % |
| 60 % | 116 091 € | 1 005 € | 15 camions | 34 % |
| 40 % | 116 332 € | 1 246 € | 15 camions | 22 % |
| 35 % (retenu) | 116 332 € | 1 246 € | 15 camions | 21 % |
| 30 % | 116 358 € | 1 272 € | 15 camions | 21 % |
| 25 % à 15 % | 116 411 € | 1 326 € | 15 camions | 16 % |

Ce qu'on en retient :
- Le premier lissage fait l'essentiel : dès 100 %, pour 1 005 €, le jour le plus chargé passe de 33 à 17 camions et l'irrégularité de 120 % à 55 %.

- De 100 % à 60 %, la régularité est gratuite : l'irrégularité passe de 55 % à 34 % sans surcoût.

- De 60 % à 35 %, elle coûte 241 € pour 13 points, soit environ 19 € par point.

- En dessous, 25 % gagne encore 5 points (16 %) pour 80 € de plus ; au-delà, plus rien ne change.

Pourquoi 35 % : l'irrégularité moyenne tombe à 21 %, soit environ 1 camion d'écart par jour pour une usine qui en reçoit 6, pour un surcoût de lissage de 1 246 € (1,1 % du coût total). 25 % serait aussi un choix raisonnable : 80 € de plus pour des semaines encore un peu plus régulières.

### Bloc 4 : report et planning final

Rôle : réduire l'avance des livraisons sans ajouter de camion, puis construire le planning complet.

Report : un camion part le dernier jour fixe avant son premier besoin, mais il transporte souvent des lignes dont le besoin tombe le jour fixe suivant ou après. Ces lignes peuvent partir avec le camion du jour fixe suivant, s'il existe et s'il a la place. On n'ajoute jamais de camion. Ensuite, si un couple a plusieurs camions le même jour qui tiennent désormais dans un seul, on les regroupe. Le report n'est gardé pour un couple que s'il ne coûte rien, car le prix dépend du poids de chaque camion, et si le couple livre toujours au moins une semaine sur tous ses jours fixes (C11).

```
Couple au profil « mardi + vendredi »
camion du mardi 17/11 : besoins du 17/11, 18/11, 20/11, 23/11
  → les lignes du 20/11 et du 23/11 peuvent partir avec le camion du vendredi 20/11 (s'il a la place)
```

Planning final : on recalcule le contenu, le poids et le coût de chaque camion, puis on ajoute les camions réservés de la quarantaine. Le résultat est une ligne par camion (planning) et une ligne par ligne de besoin, avec son camion (contenu).

In [ ]:
def prochain_jour_fixe(date, profil):
    # Premier jour du profil strictement après `date`
    return min(date + pd.Timedelta(days=(j - date.day_of_week - 1) % 7 + 1) for j in profil)


def construire_planning(choix):
    # Camions d'un choix de profils (une ligne par camion, avec la liste de ses lignes)
    return pd.concat([scenarios[(p, s, prof)].assign(plant=p, supplier_id=s, profil=[prof] * len(scenarios[(p, s, prof)]))
                      for (p, s), prof in choix.items()], ignore_index=True)


def reporter_lignes(camions):
    # Report des lignes sur le jour fixe suivant + regroupement des camions d'un même jour, couple par couple,
    # gardé seulement s'il ne coûte rien
    besoin, poids, volume = besoins["need_date"], besoins["weight"], besoins["volume"]
    charge_ok = lambda ls: poids[ls].sum() <= POIDS_MAX + 1e-9 and volume[ls].sum() <= VOLUME_MAX + 1e-9
    resultat, journal = [], []
    for (plant, supplier), groupe in camions.groupby(["plant", "supplier_id"], sort=False):
        profil = groupe["profil"].iloc[0]
        cout = lambda ls: cout_camion(poids[ls].sum(), supplier)
        cam = [{"date": r.date, "lignes": list(r.lignes)} for r in groupe.sort_values("date").itertuples()]
        cout_avant, nb_avant, reportees = sum(cout(c["lignes"]) for c in cam), len(cam), 0
        for c in cam:                                               # 1) report vers le jour fixe suivant
            suivant = prochain_jour_fixe(c["date"], profil)
            cibles = [u for u in cam if u["date"] == suivant]
            for l in sorted((l for l in c["lignes"] if besoin[l] >= suivant), key=lambda l: -poids[l]):
                for u in cibles:
                    if charge_ok(u["lignes"] + [l]):
                        c["lignes"].remove(l)
                        u["lignes"].append(l)
                        reportees += 1
                        break
        for date in sorted({c["date"] for c in cam}):                # 2) regroupement des camions d'un même jour
            du_jour = [c for c in cam if c["date"] == date and c["lignes"]]
            for a, b in itertools.combinations(du_jour, 2):
                if a["lignes"] and b["lignes"] and charge_ok(a["lignes"] + b["lignes"]):
                    a["lignes"] += b["lignes"]
                    b["lignes"] = []
        apres = [c for c in cam if c["lignes"]]
        cout_apres = sum(cout(c["lignes"]) for c in apres)
        moins_cher = cout_apres <= cout_avant + 0.005
        garde = moins_cher and profil_utilise([c["date"] for c in apres], profil)   # C11 toujours respectée
        if not garde:
            apres = [{"date": r.date, "lignes": list(r.lignes)} for r in groupe.itertuples()]
        journal.append({"plant": plant, "supplier_id": supplier, "lignes_reportees": reportees if garde else 0,
                        "camions_en_moins": nb_avant - len(apres), "gain": round(cout_avant - cout_apres, 2) if garde else 0.0,
                        "report_refuse_car_plus_cher": not moins_cher and reportees > 0,
                        "report_refuse_profil_incomplet": moins_cher and not garde})
        resultat += [{**c, "plant": plant, "supplier_id": supplier, "profil": profil} for c in apres]
    return pd.DataFrame(resultat), pd.DataFrame(journal)


def finaliser(camions):
    # Planning (une ligne par camion) et contenu (une ligne par ligne de besoin) à partir des camions
    camions = camions.sort_values(["plant", "date", "supplier_id"]).reset_index(drop=True)
    camions["camion_id"] = range(1, len(camions) + 1)
    affectation = camions[["camion_id", "date", "lignes"]].explode("lignes").set_index("lignes")
    lignes = besoins.join(affectation.rename(columns={"date": "date_livraison"}))
    lignes["avance_jours"] = (lignes["need_date"] - lignes["date_livraison"]).dt.days
    lignes.insert(0, "id_ligne", "L" + lignes.index.astype(str))
    contenu = lignes.groupby("camion_id").agg(poids=("weight", "sum"), volume=("volume", "sum"), nb_lignes=("weight", "size"))
    planning = camions.drop(columns=[c for c in ["lignes", "poids", "volume", "cout"] if c in camions]).join(contenu, on="camion_id")
    planning["cout"] = [cout_camion(p, s) for p, s in zip(planning["poids"], planning["supplier_id"])]
    planning["jours_fixes"] = planning["profil"].map(nom_profil)
    planning["reserve"] = False
    return planning, lignes


# ---------- Planning de chaque étape (pour le bilan) ----------
plannings = {}
for nom in VARIANTES:
    plannings[nom] = finaliser(construire_planning(choix_initiaux[nom]))
    plannings[f"{nom} + lissage"] = finaliser(construire_planning(choix_lisses[nom]))
camions_report, journal_report = reporter_lignes(construire_planning(choix_lisses[PRINCIPALE]))
plannings[f"{PRINCIPALE} + lissage + report"] = finaliser(camions_report)
planning, lignes_planning = plannings[f"{PRINCIPALE} + lissage + report"]
_, lignes_avant_report = plannings[f"{PRINCIPALE} + lissage"]

# ---------- Planning complet : camions optimisés + camions réservés (numérotés à la suite) ----------
decalage = int(planning["camion_id"].max())
reserves = camions_reserves.assign(camion_id=camions_reserves["numero"] + decalage, reserve=True, profil=None,
                                   jours_fixes="réservé : " + camions_reserves["motif"])
planning_complet = (pd.concat([planning, reserves.drop(columns=["numero", "motif"])], ignore_index=True)
                    .sort_values(["plant", "date", "supplier_id"]).reset_index(drop=True))
planning_complet["jour"] = planning_complet["date"].dt.day_of_week.map(dict(enumerate(JOURS)))
planning_complet["semaine"] = planning_complet["date"].dt.isocalendar().week
lignes_complet = pd.concat([lignes_planning.assign(reserve=False),
                            lignes_reserves.assign(camion_id=lignes_reserves["numero"] + decalage, reserve=True)
                            .drop(columns="numero")], ignore_index=True)

print(f"Report : {journal_report['lignes_reportees'].sum()} lignes reportées sur le jour fixe suivant "
      f"({journal_report['lignes_reportees'].gt(0).sum()} couples), {journal_report['camions_en_moins'].sum()} camion(s) en moins, "
      f"gain {eur(journal_report['gain'].sum())} ; refusé car plus cher pour {journal_report['report_refuse_car_plus_cher'].sum()} couples, "
      f"car il casserait C11 pour {journal_report['report_refuse_profil_incomplet'].sum()}")
print(f"Camions dans la semaine ajoutée (avant le {DEBUT:%d/%m}) : {(planning['date'] < DEBUT).sum()}")
print(f"Avance moyenne d'une ligne : {nombre(lignes_avant_report['avance_jours'].mean())} j -> "
      f"{nombre(lignes_planning['avance_jours'].mean())} j ; lignes livrées le jour de leur besoin : "
      f"{(lignes_avant_report['avance_jours'] == 0).sum()} -> {(lignes_planning['avance_jours'] == 0).sum()}")
print(f"Planning complet : {len(planning_complet)} camions ({len(planning)} optimisés + {NB_RESERVE} réservés), "
      f"{eur(planning_complet['cout'].sum())}")
print("Couples par nombre de jours fixes :",
      planning.drop_duplicates(["plant", "supplier_id"])["profil"].map(len).value_counts().sort_index()
      .rename(lambda n: f"{n} jour(s)").to_dict())

In [ ]:
# Avant lissage : profils les moins chers (bloc 1) ; après lissage : planning final (lissage + report)
# Tous les camions reçus sont comptés, camions réservés compris
heatmap({"avant lissage": charges_camions(avec_reserves(plannings[PRINCIPALE][0])),
         "après lissage": charges_camions(planning_complet)},
        f"{PRINCIPALE} : camions reçus par semaine et par jour (camions réservés compris)")

In [ ]:
# Détail du planning d'une usine (une ligne par camion) : changer l'usine ici
USINE = "PLT_004"
(planning_complet.loc[planning_complet["plant"] == USINE,
                      ["camion_id", "supplier_id", "jours_fixes", "date", "jour", "semaine", "reserve",
                       "nb_lignes", "poids", "volume", "cout"]]
 .round({"poids": 1, "volume": 2}))

Résultat : le report fait partir 172 lignes (32 couples) avec le camion du jour fixe suivant, au lieu de 3 à 7 jours en avance. Les lignes livrées le jour même de leur besoin passent de 247 à 413. Le gain en coût est faible (215 €) et aucun camion n'est supprimé : un camion part toujours le dernier jour fixe avant son premier besoin, donc il ne peut jamais être reporté entier sans retard. Le report sert surtout à réduire le stock des usines.

Le planning final compte 787 camions (653 optimisés + 134 réservés) pour 116 118 €. 172 couples livrent sur 1 jour fixe, et 1 sur 2 jours fixes (SUP_00117 → PLT_004, lundi + mercredi). Tous les camions partent un jour fixe de leur couple ; 57 partent dans la semaine ajoutée avant le planning (S45), pour des besoins de la première semaine.

Comparaison des deux heatmaps (avant lissage en haut, après en bas) :
- Avant lissage, chaque couple a pris son jour le moins cher, sans regarder les autres couples de l'usine : les camions s'accumulent sur certains jours. Les 20 semaines contrôlées sont hors tolérance, avec des pics comme le vendredi S47 de PLT_002 (33 camions, contre 4 le mercredi) ou les lundis de PLT_001 (20 camions en S46).

- Après lissage, les semaines deviennent régulières : la même semaine S47 de PLT_002 passe de 6 - 5 - 4 - 7 - 33 à 11 - 12 - 13 - 11 - 13 camions. Dans les semaines contrôlées, aucun jour ne dépasse 15 camions.

- La première semaine n'est plus un problème : les livraisons anticipées partent dans la semaine S45, et la S46 est régulière dans les 4 usines (PLT_001 : 10 - 7 - 8 - 8 - 8, PLT_002 : 8 - 7 - 7 - 7 - 5).

- Il ne reste que 2 semaines hors tolérance sur 20, deux petites semaines : la S50 de PLT_002 (6 camions, dont 4 le mardi) et la S49 de PLT_003 (11 camions, 3 - 2 - 3 - 2 - 1, juste au-dessus de la tolérance).

## 5. Validation des résultats

On vérifie le planning final avec les fonctions de la partie 1 (une ligne par contrainte), puis la cohérence du planning : chaque camion contient bien ce qu'il annonce, le poids livré est égal au poids demandé, et le coût total est bien la somme des coûts des camions.

In [ ]:
controle_contraintes = verifier_contraintes(planning_complet, lignes_complet, IDS_ATTENDUS)
controle_contraintes

In [ ]:
# Cas impossibles notifiés (C8, C9 : règle absolue des camions réservés, besoin d'un jour > 24,5 t)
# et semaines hors tolérance (C10)
alertes_planning = alertes(planning_complet, lignes_complet)
print(f"Alertes : {len(alertes_planning)} cas où une règle ne peut pas être respectée")
display(alertes_planning.groupby(["regle", "cause"]).size().rename("cas").to_frame())
display(alertes_planning[alertes_planning["regle"].str.startswith("C9")].drop(columns="a_faire"))
c10_lissage(planning_complet, lignes_complet).style.format({"ecart": pourcent})

In [ ]:
def valider_resultats(planning, lignes):
    # Cohérence du planning final : contenu des camions, poids livré, coût
    opti, lignes_opti = optimises(planning), lignes[~lignes["reserve"]]
    contenu = lignes_opti.groupby("camion_id").agg(poids=("weight", "sum"), volume=("volume", "sum"), nb=("weight", "size"))
    p = opti.set_index("camion_id")
    controles = {
        "Chaque camion optimisé contient au moins une ligne": opti["camion_id"].isin(contenu.index).all(),
        "Poids de chaque camion = somme des poids de ses lignes": np.allclose(p["poids"], contenu.loc[p.index, "poids"]),
        "Volume de chaque camion = somme des volumes de ses lignes": np.allclose(p["volume"], contenu.loc[p.index, "volume"]),
        "Poids total livré = poids total des besoins": np.isclose(lignes_opti["weight"].sum(), besoins["weight"].sum()),
        "Un camion réservé par ligne sans poids": (lignes[lignes["reserve"]].groupby("camion_id").size() == 1).all()
                                                   and planning["reserve"].sum() == NB_RESERVE,
        "Coût de chaque camion = grille tarifaire": np.allclose(planning["cout"], [cout_camion(w, s) for w, s in zip(planning["poids"], planning["supplier_id"])]),
        "Coût total = somme des coûts des camions": np.isclose(planning["cout"].sum(), cout_total(planning)),
    }
    return pd.DataFrame({"Contrôle": list(controles), "Résultat": ["OK" if v else "ÉCHEC" for v in controles.values()]}).set_index("Contrôle")


validation = valider_resultats(planning_complet, lignes_complet)
assert (validation["Résultat"] == "OK").all(), "le planning final est incohérent"
assert (controle_contraintes.loc[controle_contraintes["Type"] == "dure", "Violations"] == 0).all(), "une contrainte dure est violée"
validation

Résultat : toutes les contraintes dures sont respectées et le planning est cohérent.
- C7 et C8 : chaque camion construit par le moteur part un jour fixe de son couple, sans exception. 57 camions partent dans la semaine ajoutée avant le planning (S45, du 02/11 au 06/11), jamais plus de 7 jours avant le besoin. Seuls les camions réservés (lignes sans poids) partent le jour du besoin : 110 d'entre eux tombent hors des jours fixes du couple (dont 26 pour des couples qui n'ont que des lignes sans poids, donc pas de jours fixes) ; ils sont notifiés.

- C9 : en comptant tous les camions, 29 jours où un couple reçoit 2 camions ou plus, tous impossibles à éviter et notifiés :
  - 4 jours pour SUP_00037 → PLT_002 et SUP_00046 → PLT_004, dont le besoin d'un même jour dépasse 24,5 t (25,7 t et 46,7 t) ;
  - 25 jours où une ligne sans poids a son camion réservé le même jour qu'un autre camion du couple (règle absolue).

Au total, 139 alertes sont écrites dans `data/gold/alertes.csv` (partie 7), chacune avec sa cause et l'action à mener : pour les lignes sans poids, obtenir leur poids auprès du client.

- C11 : le seul couple à 2 jours fixes (SUP_00117 → PLT_004, lundi + mercredi) livre ses deux jours la même semaine.

- C10 (souple) : 2 semaines hors tolérance sur 20, deux petites semaines : la S50 de PLT_002 (6 camions, dont 4 le mardi) et la S49 de PLT_003 (11 camions, 3 - 2 - 3 - 2 - 1).

## 6. Économies

Comparaison du planning avec la baseline du milestone 2 : d'abord couple par couple, sur les couples qui coûtent le plus cher aujourd'hui, puis au total, avec et sans les lignes de la quarantaine. Les 134 camions réservés (lignes sans poids) sont les mêmes dans la baseline et dans le planning : ils s'ajoutent au coût, sans changer l'économie en euros.

In [ ]:
# Coût par couple fournisseur × usine : baseline vs algorithme retenu + lissage (camions réservés exclus, identiques dans les deux)
VERSION = f"{PRINCIPALE} + lissage"
SCENARIOS = {"Baseline": camions_base, VERSION: plannings[VERSION][0]}
COULEURS = {"Baseline": "#9ec5d6", VERSION: "royalblue"}

cout_couple = pd.DataFrame({nom: cam.groupby(["supplier_id", "plant"])["cout"].sum() for nom, cam in SCENARIOS.items()})
cout_couple.index = [f"{s} × {p}" for s, p in cout_couple.index]
top = cout_couple.nlargest(10, "Baseline").sort_values("Baseline")

ax = top.plot(kind="barh", color=[COULEURS[n] for n in top.columns], edgecolor="black", figsize=(9, 7))
for conteneur in ax.containers:
    ax.bar_label(conteneur, fmt="%.0f €", fontsize=8, padding=2)
ax.set_title("Coût par couple fournisseur × usine (10 plus chers de la baseline)")
ax.set_xlabel("Coût total sur la période (€)")
ax.margins(x=0.12)
ax.grid(axis="x", alpha=0.5)
plt.show()

part_base = top["Baseline"].sum() / cout_couple["Baseline"].sum()
print(f"Ces 10 couples : {top['Baseline'].sum():,.0f} € -> {top[VERSION].sum():,.0f} € "
      f"({100 * (1 - top[VERSION].sum() / top['Baseline'].sum()):.1f} % d'économie), "
      f"soit {part_base:.1%} du coût de la baseline")

In [ ]:
# Ce que coûtent les données manquantes : même calcul sans les lignes de la quarantaine (silver seul)
couples_quarantaine = set(map(tuple, besoins.loc[besoins.index >= N_SILVER, ["plant", "supplier_id"]].values))
FOND_VIDE = {plant: np.zeros((NB_SEMAINES, 5)) for plant in USINES}
choix_sans = {cle: prof for cle, prof in choix_initiaux[PRINCIPALE].items() if cle not in couples_quarantaine}
choix_sans, _ = lisser(choix_sans, VARIANTES[PRINCIPALE], fond=FOND_VIDE)
camions_sans, _ = reporter_lignes(construire_planning(choix_sans))
planning_sans, _ = finaliser(camions_sans)
planning_sans = planning_sans[planning_sans["camion_id"].notna()]
base_sans = camions_base[camions_base["source"] == "silver"]
COUT_BASELINE_TOTAL = COUT_BASELINE + COUT_RESERVE

qualite = pd.DataFrame({
    "Sans les lignes de la quarantaine": [base_sans["cout"].sum(), planning_sans["cout"].sum()],
    "Avec (camions réservés compris)": [COUT_BASELINE_TOTAL, planning_complet["cout"].sum()],
}, index=["Baseline", "Planning final"])
qualite.loc["Économie"] = qualite.loc["Baseline"] - qualite.loc["Planning final"]
qualite.loc["Économie (%)"] = qualite.loc["Économie"] / qualite.loc["Baseline"]
qualite.style.format(lambda v: pourcent(v) if abs(v) < 1 else eur(v))

Résultat :

- Couple par couple : les 10 couples les plus chers de la baseline pèsent 26 % de son coût. Leur coût passe de 37 349 € à 15 916 € (-57,4 %), bien plus que la moyenne. 9 de ces 10 couples sont aujourd'hui livrés 3 à 5 jours par semaine (2 en moyenne pour l'ensemble des couples) : regrouper leurs livraisons sur un jour fixe est ce qui rapporte le plus.

- Au total, le planning final coûte 116 118 € au lieu de 177 321 € : le moteur économise 61 203 € sur les 6 semaines, soit -34,5 %, en passant de 1 741 à 787 camions (divisés par 2,2).

- Les jours fixes et le lissage coûtent 6 605 € par rapport à l'algorithme de base sans contrainte de jours (109 513 €, voir le bloc 1) : on garde 90 % de l'économie maximale, avec des jours de livraison fixes et des usines équilibrées. L'algorithme mathématique, à 124 107 € (-30,0 %), est comparé dans la partie 2.

- Sans les lignes de la quarantaine, l'économie monte à -43,0 % (142 965 € -> 81 546 €). L'écart vient des 134 camions réservés (33 506 €, 29 % du coût final) : sans le poids de ces lignes, on ne peut pas les consolider. Retrouver ces poids auprès du client est le plus gros levier d'économie restant.

## 7. Sorties

Les livrables du moteur sont affichés ci-dessous et écrits en CSV dans `data/gold/`. Les alertes de la partie 5 (cas où une règle ne peut pas être respectée) y sont aussi écrites, dans `alertes.csv`.

### Sortie 1 : le profil à appliquer pour chaque couple

Une ligne par couple fournisseur × usine (`data/gold/sortie1_profils.csv`) :

| Colonne | Contenu |
|---|---|
| Fournisseur, Usine | le couple |
| Distance | distance du fournisseur au point de consolidation (inconnue pour les fournisseurs sans distance) |
| Livraisons / sem. actuelles | jours de livraison par semaine dans la baseline, en moyenne sur les semaines où le couple est livré |
| Fréquence cible | nombre de jours fixes du profil retenu (1 ou 2 par semaine) |
| Jour | jour(s) fixe(s) de livraison |
| Camions / sem. | camions par semaine dans le planning final, en moyenne sur les semaines où le couple est livré |
| Remplissage poids | poids moyen d'un camion du couple, en % des 24,5 t |

Les camions réservés (lignes sans poids) ne sont pas comptés : ils partent le jour du besoin, hors profil.

In [ ]:
# ---------- Sortie 1 : le profil à appliquer pour chaque couple ----------
distance_brute = distances.where(~distances.index.isin(SANS_DISTANCE))   # NaN = distance inconnue


def moyennes_par_semaine(camions):
    # Par couple, sur les semaines où il est livré : jours de livraison et camions par semaine (moyennes)
    par_sem = camions.assign(semaine=camions["date"].dt.isocalendar().week).groupby(["plant", "supplier_id", "semaine"])
    jours = par_sem["date"].nunique().groupby(["plant", "supplier_id"]).mean()
    nb_camions = par_sem.size().groupby(["plant", "supplier_id"]).mean()
    return jours, nb_camions


livraisons_actuelles, _ = moyennes_par_semaine(camions_base)
_, camions_semaine = moyennes_par_semaine(planning)
par_couple = planning.groupby(["plant", "supplier_id"])
sortie1 = pd.DataFrame({
    "frequence_cible": par_couple["profil"].first().map(len),
    "jours": par_couple["jours_fixes"].first(),
    "camions_semaine": camions_semaine.round(2),
    "remplissage_poids": (par_couple["poids"].mean() / POIDS_MAX).round(4),
}).reset_index()
sortie1.insert(2, "distance_km", sortie1["supplier_id"].map(distance_brute))
sortie1.insert(3, "livraisons_semaine_actuelles",
               [round(livraisons_actuelles.get((p, s), 0), 2) for p, s in zip(sortie1["plant"], sortie1["supplier_id"])])
sortie1 = sortie1.rename(columns={"supplier_id": "fournisseur", "plant": "usine"})[
    ["fournisseur", "usine", "distance_km", "livraisons_semaine_actuelles", "frequence_cible", "jours",
     "camions_semaine", "remplissage_poids"]].sort_values(["usine", "fournisseur"]).reset_index(drop=True)

assert sortie1[["fournisseur", "usine"]].duplicated().sum() == 0      # un profil par couple
assert len(sortie1) == besoins.groupby(["plant", "supplier_id"]).ngroups
SORTIE.mkdir(parents=True, exist_ok=True)
sortie1.to_csv(SORTIE / "sortie1_profils.csv", index=False)
print(f"sortie1_profils.csv : {len(sortie1)} couples ; aperçu des 10 premiers :")

(sortie1.head(10).rename(columns={"fournisseur": "Fournisseur", "usine": "Usine", "distance_km": "Distance",
                         "livraisons_semaine_actuelles": "Livraisons / sem. actuelles",
                         "frequence_cible": "Fréquence cible", "jours": "Jour", "camions_semaine": "Camions / sem.",
                         "remplissage_poids": "Remplissage poids"})
 .style.format({"Distance": lambda d: "inconnue" if pd.isna(d) else f"{d:.0f} km",
                "Livraisons / sem. actuelles": nombre, "Fréquence cible": "{} / semaine".format,
                "Camions / sem.": nombre, "Remplissage poids": pourcent})
 .hide(axis="index"))

### Sortie 2 : les materials regroupés dans chaque camion

Une ligne par camion × material × quai (`data/gold/sortie2_camions.csv`) : camion, usine, fournisseur, date de livraison, material, quai, dates des besoins regroupés, nombre de lignes de besoin, quantité et poids. Les camions réservés (une ligne sans poids chacun) sont inclus, avec un poids vide.

Le tableau ci-dessous détaille un camion : par défaut celui qui regroupe le plus de lignes de besoin, sinon celui choisi dans `CAMION`.

In [ ]:
# ---------- Sortie 2 : les materials regroupés dans chaque camion ----------
sortie2 = (lignes_complet
           .groupby(["camion_id", "plant", "supplier_id", "date_livraison", "material", "unloading_point"])
           .agg(besoins=("need_date", lambda d: ", ".join(sorted(d.dt.strftime("%Y-%m-%d").unique()))),
                nb_lignes=("need_date", "size"), quantite=("schedule_quantity", "sum"),
                poids_kg=("weight", lambda w: w.sum(min_count=1)), reserve=("reserve", "first"))
           .reset_index()
           .rename(columns={"plant": "usine", "supplier_id": "fournisseur", "unloading_point": "quai"}))
sortie2.insert(4, "jour", sortie2["date_livraison"].dt.day_of_week.map(dict(enumerate(JOURS))))
sortie2["poids_kg"] = sortie2["poids_kg"].round(1)
sortie2 = sortie2.sort_values(["camion_id", "poids_kg", "material"], ascending=[True, False, True]).reset_index(drop=True)

assert sortie2["nb_lignes"].sum() == len(IDS_ATTENDUS)                  # toutes les lignes de besoin
assert sortie2["camion_id"].nunique() == len(planning_complet)          # tous les camions
sortie2.to_csv(SORTIE / "sortie2_camions.csv", index=False, date_format="%Y-%m-%d")
print(f"sortie2_camions.csv : {len(sortie2)} lignes (camion × material × quai), {sortie2['camion_id'].nunique()} camions")
alertes_planning.to_csv(SORTIE / "alertes.csv", index=False, date_format="%Y-%m-%d")
print(f"alertes.csv : {len(alertes_planning)} cas où une règle ne peut pas être respectée")


# ---------- Détail d'un camion ----------
CAMION = None   # numéro de camion à détailler ; None = celui qui regroupe le plus de lignes de besoin
NB_AFFICHES = 10
MOIS = ["janvier", "février", "mars", "avril", "mai", "juin", "juillet", "août", "septembre", "octobre", "novembre", "décembre"]


def poids_camion(contenu):
    # Poids total d'un camion et remplissage ; camion réservé : poids inconnu
    poids = contenu["poids_kg"].sum(min_count=1)
    if pd.isna(poids):
        return "poids inconnu (camion réservé)"
    return f"{nombre(round(poids))} kg (remplissage poids {pourcent(poids / POIDS_MAX)})"


def detail_camion(camion_id, nb_affiches=NB_AFFICHES):
    contenu = sortie2[sortie2["camion_id"] == camion_id]
    dates = pd.to_datetime([d for b in contenu["besoins"] for d in b.split(", ")])
    un_mois = dates.month.nunique() == 1
    jours_besoin = lambda b: ", ".join(str(pd.Timestamp(d).day) if un_mois else pd.Timestamp(d).strftime("%d/%m")
                                       for d in b.split(", "))
    table = (contenu.head(nb_affiches)[["material", "quai", "besoins", "quantite", "poids_kg"]]
             .assign(besoins=lambda t: t["besoins"].map(jours_besoin)).reset_index(drop=True))
    reste = contenu.iloc[nb_affiches:]
    if len(reste):
        table.loc[len(table)] = [f"{len(reste)} autres materials", "", "", reste["quantite"].sum(), reste["poids_kg"].sum()]
    camion = contenu.iloc[0]
    print(f"Camion n° {camion_id} : {camion['fournisseur']} -> {camion['usine']}, {camion['jour'].lower()} "
          f"{camion['date_livraison']:%d/%m}, {len(contenu)} materials, "
          f"{contenu['nb_lignes'].sum()} lignes de besoin, {poids_camion(contenu)}")
    titre_besoins = f"Besoins regroupés (jours de {MOIS[dates[0].month - 1]})" if un_mois else "Besoins regroupés"
    return (table.rename(columns={"material": "Material", "quai": "Quai", "besoins": titre_besoins,
                                  "quantite": "Quantité", "poids_kg": "Poids (kg)"})
            .style.format({"Quantité": nombre, "Poids (kg)": lambda p: nombre(round(p))}, na_rep="–")
            .hide(axis="index"))


camion_detail = CAMION if CAMION is not None else int(planning_complet.loc[planning_complet["nb_lignes"].idxmax(), "camion_id"])
detail_camion(camion_detail)

## Conclusion

Le moteur produit le planning de livraison complet des 4 usines sur les 6 semaines, avec :
- des jours fixes pour chaque couple (172 couples sur 1 jour, 1 sur 2 jours), respectés dès la première semaine grâce à la semaine ajoutée avant le planning (S45) : aucun camion ne part hors des jours fixes de son couple 

- au plus 1 camion par couple et par jour, sauf les cas impossibles, tous notifiés dans `alertes.csv` : 2 couples dont le besoin d'un seul jour dépasse 24,5 t, et les lignes sans poids, dont le camion réservé part le jour du besoin (règle absolue) 

- des usines équilibrées : 2 semaines hors tolérance sur 20, deux petites semaines (S50 de PLT_002, S49 de PLT_003) 

- toutes les contraintes dures respectées : capacité, aucune livraison en retard, au plus 7 jours d'avance, chaque ligne livrée une fois, et un profil à 2 jours utilisé en entier au moins une semaine (C11).

Il économise 61 203 € (-34,5 %) par rapport à la situation actuelle, et -43,0 % sur les lignes dont les données sont complètes.

Les deux livrables sont écrits dans `data/gold/` (aperçus dans la partie 7) :
- Sortie 1 (`sortie1_profils.csv`) : le profil à appliquer à chacun des 173 couples. Les couples passent en moyenne de 2 jours de livraison par semaine aujourd'hui à 1 camion par semaine, sur 1 jour fixe (172 couples ; un seul couple livre sur 2 jours). Le remplissage reste faible (2,1 % du poids maximal pour le couple médian) : la plupart des couples livrent peu, et c'est le nombre de camions, pas leur taille, qui fait l'économie.

- Sortie 2 (`sortie2_camions.csv`) : les materials regroupés dans chacun des 787 camions, avec leur quai, les jours de besoin regroupés, la quantité et le poids. Le camion le plus chargé en lignes (SUP_00097 → PLT_002, vendredi 20/11) regroupe 91 lignes de besoin et 43 materials.

Limites et pistes :
- Qualité des données : 134 lignes sans poids (29 % du coût final) ne peuvent pas être consolidées. Leur camion réservé crée 135 des 139 alertes (25 jours avec 2 camions pour un couple, 110 camions hors jours fixes). Leur poids dépend du client.

- Ligne de plus de 24,5 t (SUP_00074 → PLT_003, 26,9 t le 15/12) : exclue en attendant la réponse du client.

- Seuil de lissage : 35 % et 40 % donnent le même planning ; 25 % rendrait les semaines un peu plus régulières (irrégularité de 16 % au lieu de 21 %) pour 80 € de plus.

- Profils à plusieurs jours : avec la règle du profil utilisé (C11), un couple ne garde 2 ou 3 jours fixes que s'il livre au moins une semaine sur tous ses jours. Comme un camion prend les besoins de 7 jours, presque tous les couples n'ont besoin que d'un camion par semaine : 172 couples sur 173 passent à 1 jour fixe. Sans cette règle, le planning coûterait environ 3 900 € de moins (112 252 €, 736 camions). Les profils à 3 jours restent possibles, mais aucun couple n'en a besoin.

- Lissage glouton : il change un profil à la fois. Une recherche plus large (par exemple avec un solveur) pourrait encore réduire le surcoût.

## 8. Algorithme complet

L'algorithme d'optimisation entier, des fichiers d'entrée aux deux sorties, dans une seule cellule qu'on peut exécuter d'un coup. Elle ne dépend d'aucune cellule précédente

Entrées (dans `../data/`) : `bronze/rate_sheet.csv`, `bronze/distances.csv`, `silver/released_schedule_cleaned.csv`, `silver/released_schedule_quarantine.csv` et `baseline/baseline_camion.csv`.

Sorties : `gold/sortie1_profils.csv` et `gold/sortie2_camions.csv`, identiques à celles de la partie 7, et un résumé des résultats. Les contraintes dures sont vérifiées par des `assert` : la cellule s'arrête si l'une d'elles est violée.

In [ ]:
# Algorithme d'optimisation complet : des fichiers d'entrée aux deux sorties, en une seule exécution.
# Tout est dans la fonction moteur_consolidation : la cellule ne dépend d'aucune cellule précédente.
import itertools
from pathlib import Path

import numpy as np
import pandas as pd


def moteur_consolidation(data=Path("../data"), seuil_lissage=0.35, ecrire=True):
    # ---------- Paramètres ----------
    POIDS_MAX, VOLUME_MAX = 24500, 110.4          # capacité d'un camion : kg, m³
    FENETRE = pd.Timedelta(days=7)                # avance maximale d'une livraison
    MIN_CAMIONS_SEMAINE = 5                       # semaines plus petites : pas contrôlées
    JOURS = ["Lundi", "Mardi", "Mercredi", "Jeudi", "Vendredi"]
    PROFILS = [p for n in (1, 2, 3) for p in itertools.combinations(range(5), n)]   # 1 à 3 jours fixes

    # ---------- Entrées ----------
    couts = pd.read_csv(data / "bronze" / "rate_sheet.csv", index_col="km_max")
    distances = pd.read_csv(data / "bronze" / "distances.csv", encoding="utf-8-sig").set_index("supplier_id")["distance"]
    silver = pd.read_csv(data / "silver" / "released_schedule_cleaned.csv", parse_dates=["need_date"])
    quarantaine = pd.read_csv(data / "silver" / "released_schedule_quarantine.csv", parse_dates=["need_date"])
    camions_base = pd.read_csv(data / "baseline" / "baseline_camion.csv", parse_dates=["date"])

    # ---------- Coût d'un camion ----------
    TRANCHES_KM = [int(km) for km in couts.index if not km.startswith(">")]
    TRANCHES_KG = [int(kg) for kg in couts.columns if not kg.startswith(">")]
    distance_brute = distances.copy()                                  # NaN = distance inconnue (sortie 1)
    distances = distances.fillna(TRANCHES_KM[-1] + 1)                  # distance inconnue -> tranche > 650 km

    def tranche(valeur, tranches):
        return next((str(b) for b in tranches if valeur <= b), f">{tranches[-1]}")

    def cout_camion(poids, supplier):
        poids = TRANCHES_KG[-1] + 1 if pd.isna(poids) else poids      # poids inconnu -> tranche > 4 000 kg
        base = couts.loc[tranche(distances[supplier], TRANCHES_KM), tranche(poids, TRANCHES_KG)]
        return float(round(base * (1 + 2.3923) * 1.1943, 2))          # marge 239,23 % et surcharge diesel

    # ---------- Données à consolider ----------
    sans_poids = quarantaine["weight"].isna()
    a_consolider = ~sans_poids & (quarantaine["weight"] <= POIDS_MAX)  # la ligne de plus de 24,5 t est exclue
    besoins = (pd.concat([silver, quarantaine.loc[a_consolider, silver.columns]], ignore_index=True)
               .sort_values(["plant", "supplier_id", "need_date", "weight"], ascending=[True, True, True, False])
               [["need_date", "supplier_id", "material", "plant", "unloading_point", "schedule_quantity", "weight", "volume"]])

    # Lignes sans poids : un camion réservé par ligne, le jour du besoin
    reserves = quarantaine[sans_poids].drop(columns="flags").reset_index(drop=True)
    reserves["numero"] = range(1, len(reserves) + 1)
    camions_reserves = pd.DataFrame({"plant": reserves["plant"], "supplier_id": reserves["supplier_id"],
                                     "date": reserves["need_date"],
                                     "cout": [cout_camion(np.nan, s) for s in reserves["supplier_id"]]})

    # Horizon
    DEBUT = besoins["need_date"].min()
    DEBUT_LIVRAISONS = DEBUT - pd.Timedelta(days=7)                    # semaine ajoutée avant le planning
    SEMAINES = sorted({int(w) for w in besoins["need_date"].dt.isocalendar().week} | {int(DEBUT_LIVRAISONS.isocalendar().week)})
    CONTROLE = list(range(1, len(SEMAINES) - 1))                       # semaine ajoutée et dernière : pas contrôlées
    USINES = sorted(besoins["plant"].unique())

    def charge_usine(dates):
        # Camions reçus : une ligne par semaine, une colonne par jour (lundi -> vendredi)
        charge = np.zeros((len(SEMAINES), 5))
        for d in dates:
            if d.isocalendar().week in SEMAINES and d.day_of_week < 5:
                charge[SEMAINES.index(d.isocalendar().week), d.day_of_week] += 1
        return charge

    def hors_tolerance(charge):
        # Un jour s'écarte-t-il de la moyenne de la semaine de plus du seuil (et d'au moins 1 camion) ?
        moyenne = charge.mean(axis=1, keepdims=True)
        depasse = (np.abs(charge - moyenne) > np.maximum(seuil_lissage * moyenne, 1.0) + 1e-9).any(axis=1)
        return depasse & (charge.sum(axis=1) >= MIN_CAMIONS_SEMAINE)

    # ================= Bloc 1 : jours fixes =================
    def date_camion(besoin, profil):
        # Dernier jour fixe avant ou égal au besoin (dans la semaine ajoutée pour les premiers besoins)
        return max(besoin - pd.Timedelta(days=(besoin.day_of_week - j) % 7) for j in profil)

    def profil_utilise(dates, profil):
        # C11 : au moins une semaine avec un camion sur chacun des jours fixes
        jours_par_semaine = {}
        for d in dates:
            jours_par_semaine.setdefault(tuple(d.isocalendar())[:2], set()).add(d.day_of_week)
        return any(set(profil) <= jours for jours in jours_par_semaine.values())

    def consolider_couple(lignes, profil):
        restantes, camions = list(lignes.itertuples()), []
        while restantes:
            date = date_camion(restantes[0].need_date, profil)
            poids, volume, contenu, i = 0.0, 0.0, [], 0
            while i < len(restantes):
                ligne = restantes[i]
                if ligne.need_date > date + FENETRE:
                    break                                              # hors fenêtre de 7 jours : camion fermé
                if poids + ligne.weight > POIDS_MAX or volume + ligne.volume > VOLUME_MAX:
                    i += 1                                             # ne rentre pas : ligne suivante
                    continue
                poids, volume = poids + ligne.weight, volume + ligne.volume
                contenu.append(ligne.Index)
                restantes.pop(i)
                i = 0
            camions.append({"date": date, "poids": poids, "volume": volume, "lignes": contenu})
        return pd.DataFrame(camions)

    scenarios = {}                                                     # chaque couple × chaque profil
    for (plant, supplier), lignes_couple in besoins.groupby(["plant", "supplier_id"]):
        for profil in PROFILS:
            camions = consolider_couple(lignes_couple, profil)
            camions["cout"] = [cout_camion(p, supplier) for p in camions["poids"]]
            scenarios[(plant, supplier, profil)] = camions
    cout_option = {k: c["cout"].sum() for k, c in scenarios.items()}
    nb_option = {k: len(c) for k, c in scenarios.items()}
    charge_option = {k: charge_usine(c["date"]) for k, c in scenarios.items()}
    max_jour = {k: int(c.groupby("date").size().max()) for k, c in scenarios.items()}
    utilise = {(p, s, prof): profil_utilise(c["date"], prof) for (p, s, prof), c in scenarios.items()}

    def autorises(plant, supplier):
        # Profils utilisés (C11 : une semaine au moins sur tous les jours fixes) et sans 2 camions le même jour (C9) ;
        # si aucun ne respecte C9 (besoin d'un jour > 24,5 t) : tous les profils utilisés
        utilises = [p for p in PROFILS if utilise[(plant, supplier, p)]]
        ok = [p for p in utilises if max_jour[(plant, supplier, p)] <= 1]
        return ok or utilises

    choix = {}                                                         # profil le moins cher de chaque couple
    for k in sorted((k for k in scenarios if k[2] in autorises(k[0], k[1])),
                    key=lambda k: (k[0], k[1], round(cout_option[k], 2), nb_option[k], len(k[2]), k[2])):
        choix.setdefault(k[:2], k[2])

    # ================= Blocs 2 et 3 : contrôle de l'équilibre et lissage =================
    def dispersion(charge):
        charge = charge[CONTROLE]
        return float(((charge - charge.mean(axis=1, keepdims=True)) ** 2).sum())

    nb_changements = 0
    for plant in USINES:
        charge = charge_usine(camions_reserves.loc[camions_reserves["plant"] == plant, "date"]) + sum(
            charge_option[(p, s, prof)] for (p, s), prof in choix.items() if p == plant)
        while hors_tolerance(charge[CONTROLE]).any():
            actuelle, meilleur = dispersion(charge), None
            for (p, s), prof in choix.items():
                if p != plant:
                    continue
                for prof2 in autorises(p, s):
                    if prof2 == prof:
                        continue
                    nouvelle = charge - charge_option[(p, s, prof)] + charge_option[(p, s, prof2)]
                    gain = actuelle - dispersion(nouvelle)
                    if gain <= 1e-9:
                        continue
                    cle = (max(cout_option[(p, s, prof2)] - cout_option[(p, s, prof)], 0) / gain, -gain)
                    if meilleur is None or cle < meilleur[0]:
                        meilleur = (cle, s, prof2, nouvelle)
            if meilleur is None:
                break                                                  # plus aucun changement n'améliore
            _, s, prof2, charge = meilleur
            choix[(plant, s)] = prof2                                  # changement de profil au moindre surcoût
            nb_changements += 1

    # ================= Bloc 4 : report et planning final =================
    def prochain_jour_fixe(date, profil):
        return min(date + pd.Timedelta(days=(j - date.day_of_week - 1) % 7 + 1) for j in profil)

    besoin, poids, volume = besoins["need_date"], besoins["weight"], besoins["volume"]

    def charge_ok(ls):
        return poids[ls].sum() <= POIDS_MAX + 1e-9 and volume[ls].sum() <= VOLUME_MAX + 1e-9

    camions = []
    for (plant, supplier), profil in choix.items():
        def cout(ls):
            return cout_camion(poids[ls].sum(), supplier)
        groupe = scenarios[(plant, supplier, profil)]
        cam = [{"date": r.date, "lignes": list(r.lignes)} for r in groupe.sort_values("date").itertuples()]
        cout_avant = sum(cout(c["lignes"]) for c in cam)
        for c in cam:                                                  # report vers le jour fixe suivant
            suivant = prochain_jour_fixe(c["date"], profil)
            cibles = [u for u in cam if u["date"] == suivant]
            for l in sorted((l for l in c["lignes"] if besoin[l] >= suivant), key=lambda l: -poids[l]):
                for u in cibles:
                    if charge_ok(u["lignes"] + [l]):
                        c["lignes"].remove(l)
                        u["lignes"].append(l)
                        break
        for date in sorted({c["date"] for c in cam}):                  # regroupement des camions d'un même jour
            du_jour = [c for c in cam if c["date"] == date and c["lignes"]]
            for a, b in itertools.combinations(du_jour, 2):
                if a["lignes"] and b["lignes"] and charge_ok(a["lignes"] + b["lignes"]):
                    a["lignes"] += b["lignes"]
                    b["lignes"] = []
        apres = [c for c in cam if c["lignes"]]
        if (sum(cout(c["lignes"]) for c in apres) > cout_avant + 0.005          # report gardé seulement s'il ne coûte
                or not profil_utilise([c["date"] for c in apres], profil)):   # rien et garde le profil utilisé (C11)
            apres = [{"date": r.date, "lignes": list(r.lignes)} for r in groupe.itertuples()]
        camions += [{**c, "plant": plant, "supplier_id": supplier, "profil": profil} for c in apres]

    # Planning : une ligne par camion ; contenu : une ligne par ligne de besoin
    planning = pd.DataFrame(camions).sort_values(["plant", "date", "supplier_id"]).reset_index(drop=True)
    planning["camion_id"] = range(1, len(planning) + 1)
    affectation = planning[["camion_id", "date", "lignes"]].explode("lignes").set_index("lignes")
    lignes = besoins.join(affectation.rename(columns={"date": "date_livraison"}))
    contenu = lignes.groupby("camion_id").agg(poids=("weight", "sum"), volume=("volume", "sum"))
    planning = planning.drop(columns="lignes").join(contenu, on="camion_id")
    planning["cout"] = [cout_camion(p, s) for p, s in zip(planning["poids"], planning["supplier_id"])]
    planning["jours_fixes"] = planning["profil"].map(lambda prof: " + ".join(JOURS[j] for j in prof))

    # Camions réservés, numérotés à la suite
    decalage = len(planning)
    lignes_reserves = reserves.assign(date_livraison=reserves["need_date"], camion_id=reserves["numero"] + decalage)
    lignes_complet = pd.concat([lignes.assign(reserve=False), lignes_reserves.assign(reserve=True)], ignore_index=True)
    planning_complet = pd.concat([planning.assign(reserve=False),
                                  camions_reserves.assign(camion_id=reserves["numero"] + decalage, reserve=True)],
                                 ignore_index=True)

    # ---------- Contrôles des contraintes dures ----------
    avance = (lignes_complet["need_date"] - lignes_complet["date_livraison"]).dt.days
    assert (planning["poids"] <= POIDS_MAX + 1e-6).all() and (planning["volume"] <= VOLUME_MAX + 1e-6).all()   # C1, C2
    assert lignes_complet["camion_id"].notna().all() and len(lignes_complet) == len(besoins) + len(reserves)   # C3
    assert avance.between(0, 7).all()                                  # C5, C6 : jamais de retard, 7 jours d'avance max
    assert planning_complet["date"].dt.day_of_week.lt(5).all()         # C7 : jours ouvrés
    assert (planning_complet["date"] >= DEBUT_LIVRAISONS).all()         # C7 : pas avant la semaine ajoutée
    assert all(d.day_of_week in prof for d, prof in zip(planning["date"], planning["profil"]))   # C8 : un jour fixe
    assert all(profil_utilise(g["date"], g["profil"].iloc[0])
               for _, g in planning.groupby(["plant", "supplier_id"]))  # C11 : profil utilisé

    # ---------- Alertes : cas impossibles à notifier (C9 : 2 camions un même jour ; C8 : camion réservé hors jours fixes) ----------
    poids_lourd, sans_poids = "besoin d'un jour > 24,5 t", "ligne sans poids (camion réservé)"
    a_faire = {poids_lourd: "aucune : le besoin du jour dépasse la capacité d'un camion, 2 camions sont nécessaires",
               sans_poids: "obtenir le poids de la ligne auprès du client pour pouvoir la consolider"}
    besoin_jour = besoins.groupby(["plant", "supplier_id", "need_date"])["weight"].sum()
    lourds = set(besoin_jour[besoin_jour > POIDS_MAX].index.droplevel("need_date"))
    par_jour = planning_complet.groupby(["plant", "supplier_id", "date"]).agg(camions=("reserve", "size"),
                                                                              reserves=("reserve", "sum")).reset_index()
    cas = []
    for r in par_jour[par_jour["camions"] > 1].itertuples():
        lourd = r.camions - r.reserves > 1
        assert not lourd or (r.plant, r.supplier_id) in lourds                # C9 : sinon, 2 camions évitables
        causes = ([poids_lourd] if lourd else []) + ([sans_poids] if r.reserves else [])
        cas.append({"regle": "C9 : 2 camions ou plus le même jour", "usine": r.plant, "fournisseur": r.supplier_id,
                    "date": r.date, "camions": int(r.camions), "cause": " + ".join(causes),
                    "a_faire": " ; ".join(a_faire[m] for m in causes)})
    profils = planning.groupby(["plant", "supplier_id"])["profil"].first()
    for r in camions_reserves.itertuples():
        profil = profils.get((r.plant, r.supplier_id))
        if profil is None or r.date.day_of_week not in profil:
            cas.append({"regle": "C8 : camion hors jours fixes", "usine": r.plant, "fournisseur": r.supplier_id,
                        "date": r.date, "camions": 1,
                        "cause": sans_poids + ("" if profil is not None else ", couple sans autre livraison (pas de jours fixes)"),
                        "a_faire": a_faire[sans_poids]})
    alertes = (pd.DataFrame(cas, columns=["regle", "usine", "fournisseur", "date", "camions", "cause", "a_faire"])
               .sort_values(["regle", "usine", "date", "fournisseur"]).reset_index(drop=True))

    # ================= Sortie 1 : le profil à appliquer pour chaque couple =================
    def moyennes_par_semaine(c):
        par_sem = c.assign(semaine=c["date"].dt.isocalendar().week).groupby(["plant", "supplier_id", "semaine"])
        return (par_sem["date"].nunique().groupby(["plant", "supplier_id"]).mean(),
                par_sem.size().groupby(["plant", "supplier_id"]).mean())

    livraisons_actuelles, _ = moyennes_par_semaine(camions_base[~camions_base["reserve"]])
    _, camions_semaine = moyennes_par_semaine(planning)
    par_couple = planning.groupby(["plant", "supplier_id"])
    sortie1 = pd.DataFrame({"frequence_cible": par_couple["profil"].first().map(len),
                            "jours": par_couple["jours_fixes"].first(),
                            "camions_semaine": camions_semaine.round(2),
                            "remplissage_poids": (par_couple["poids"].mean() / POIDS_MAX).round(4)}).reset_index()
    sortie1.insert(2, "distance_km", sortie1["supplier_id"].map(distance_brute))
    sortie1.insert(3, "livraisons_semaine_actuelles",
                   [round(livraisons_actuelles.get((p, s), 0), 2) for p, s in zip(sortie1["plant"], sortie1["supplier_id"])])
    sortie1 = (sortie1.rename(columns={"supplier_id": "fournisseur", "plant": "usine"})
               [["fournisseur", "usine", "distance_km", "livraisons_semaine_actuelles", "frequence_cible", "jours",
                 "camions_semaine", "remplissage_poids"]].sort_values(["usine", "fournisseur"]).reset_index(drop=True))

    # ================= Sortie 2 : les materials regroupés dans chaque camion =================
    sortie2 = (lignes_complet
               .groupby(["camion_id", "plant", "supplier_id", "date_livraison", "material", "unloading_point"])
               .agg(besoins=("need_date", lambda d: ", ".join(sorted(d.dt.strftime("%Y-%m-%d").unique()))),
                    nb_lignes=("need_date", "size"), quantite=("schedule_quantity", "sum"),
                    poids_kg=("weight", lambda w: w.sum(min_count=1)), reserve=("reserve", "first"))
               .reset_index()
               .rename(columns={"plant": "usine", "supplier_id": "fournisseur", "unloading_point": "quai"}))
    sortie2.insert(4, "jour", sortie2["date_livraison"].dt.day_of_week.map(dict(enumerate(JOURS))))
    sortie2["poids_kg"] = sortie2["poids_kg"].round(1)
    sortie2 = sortie2.sort_values(["camion_id", "poids_kg", "material"], ascending=[True, False, True]).reset_index(drop=True)

    if ecrire:
        (data / "gold").mkdir(parents=True, exist_ok=True)
        sortie1.to_csv(data / "gold" / "sortie1_profils.csv", index=False)
        sortie2.to_csv(data / "gold" / "sortie2_camions.csv", index=False, date_format="%Y-%m-%d")
        alertes.to_csv(data / "gold" / "alertes.csv", index=False, date_format="%Y-%m-%d")

    # ---------- Résumé ----------
    base = camions_base[~camions_base["reserve"]]                      # camions réservés : les mêmes des deux côtés
    cout_baseline = sum(cout_camion(p, s) for p, s in zip(base["poids"], base["supplier_id"])) + camions_reserves["cout"].sum()
    cout_final = planning_complet["cout"].sum()
    charges = {u: charge_usine(planning_complet.loc[planning_complet["plant"] == u, "date"])[CONTROLE] for u in USINES}
    resume = {
        "Couples": len(sortie1),
        "Couples sur 1 / 2 / 3 jours fixes": " / ".join(str((sortie1["frequence_cible"] == n).sum()) for n in (1, 2, 3)),
        "Changements de profil (lissage)": nb_changements,
        "Camions (dont réservés)": f"{len(planning_complet)} ({len(camions_reserves)})",
        "Camions baseline": len(camions_base),
        "Coût baseline": f"{cout_baseline:,.0f} €".replace(",", " "),
        "Coût final": f"{cout_final:,.0f} €".replace(",", " "),
        "Économie": f"{cout_baseline - cout_final:,.0f} € (-{(cout_baseline - cout_final) / cout_baseline:.1%})".replace(",", " ").replace(".", ",").replace("%", " %"),
        "Semaines hors tolérance": f"{sum(int(hors_tolerance(c).sum()) for c in charges.values())} / {len(USINES) * len(CONTROLE)}",
        "Alertes (cas impossibles notifiés)": f"{len(alertes)} (C9 : {alertes['regle'].str.startswith('C9').sum()}, "
                                              f"C8 : {alertes['regle'].str.startswith('C8').sum()})",
    }
    return sortie1, sortie2, planning_complet, pd.Series(resume, name="Résultat")


sortie1_finale, sortie2_finale, planning_final, resume = moteur_consolidation()
display(resume.to_frame())